Code for saving out model ensemble means for CMIP6 WUS winter warming paper.

In [3]:
import netCDF4
#import h5netcdf
import xarray as xr
import scipy
import os
#os.environ['ESMFMKFILE'] = '/home/tessj/miniconda3/envs/xarrayxesmf/lib/esmf.mk'
import xesmf as xe
import numpy as np
#import xeofs as xeo
import scipy.stats as stats
import warnings
import matplotlib.pyplot as plt
import scipy.signal as signal
import pandas as pd
import cartopy.crs as ccrs
import cartopy
import seaborn as sns
import statsmodels.api as sm
import json

def fix_coords_lon(ds):
    ds = ds.assign_coords(longitude=(((ds.longitude + 180) % 360) - 180)).sortby(['longitude','latitude'])
    return ds

def fix_coords_lon_lat(ds):
    ds = ds.assign_coords(lon=(((ds.lon + 180) % 360) - 180)).sortby(['lon','lat'])
    return ds

## open model ensemble members

In [4]:
with open('/home/tessj/wus_temp_trends/modmembers_rlut_hist.json', 'r') as file:
        modmembers_rlut_hist = json.load(file)
with open('/home/tessj/wus_temp_trends/modmembers_rlus_hist.json', 'r') as file:
        modmembers_rlus_hist = json.load(file)
with open('/home/tessj/wus_temp_trends/modmembers_rlds_hist.json', 'r') as file:
        modmembers_rlds_hist = json.load(file)

In [5]:
with open('/home/tessj/wus_temp_trends/modmembers_rlut_ssp245.json', 'r') as file:
        modmembers_rlut_ssp245 = json.load(file)
with open('/home/tessj/wus_temp_trends/modmembers_rlus_ssp245.json', 'r') as file:
        modmembers_rlus_ssp245 = json.load(file)
with open('/home/tessj/wus_temp_trends/modmembers_rlds_ssp245.json', 'r') as file:
        modmembers_rlds_ssp245 = json.load(file)
        
with open('/home/tessj/wus_temp_trends/modmembers_huss_ssp245.json', 'r') as file:
        modmembers_huss_ssp245 = json.load(file)
with open('/home/tessj/wus_temp_trends/modmembers_huss_hist.json', 'r') as file:
        modmembers_huss_hist = json.load(file)

with open('/home/tessj/wus_temp_trends/modmembers_hus300_ssp245.json', 'r') as file:
        modmembers_hus300_ssp245 = json.load(file)
with open('/home/tessj/wus_temp_trends/modmembers_hus300_hist.json', 'r') as file:
        modmembers_hus300_hist = json.load(file)

In [6]:
modmembesnss_hfls_ssp245 = {}
modmembers_hfss_ssp245 = {}
modmembers_clt_ssp245 = {}
modmembers_rsds_ssp245 = {}
modmembers_rsus_ssp245 = {}
modmembers_rsdt_ssp245 = {}
modmembers_rsut_ssp245 = {}

with open('/home/tessj/wus_temp_trends/modmembers_hfls_ssp245.json', 'r') as file:
        modmembers_hfls_ssp245 = json.load(file)
with open('/home/tessj/wus_temp_trends/modmembers_hfss_ssp245.json', 'r') as file:
        modmembers_hfss_ssp245 = json.load(file)
with open('/home/tessj/wus_temp_trends/modmembers_clt_ssp245.json', 'r') as file:
        modmembers_clt_ssp245 = json.load(file)
with open('/home/tessj/wus_temp_trends/modmembers_rsds_ssp245.json', 'r') as file:
        modmembers_rsds_ssp245 = json.load(file)
with open('/home/tessj/wus_temp_trends/modmembers_rsus_ssp245.json', 'r') as file:
        modmembers_rsus_ssp245 = json.load(file)
with open('/home/tessj/wus_temp_trends/modmembers_rsdt_ssp245.json', 'r') as file:
        modmembers_rsdt_ssp245 = json.load(file)
with open('/home/tessj/wus_temp_trends/modmembers_rsut_ssp245.json', 'r') as file:
        modmembers_rsut_ssp245 = json.load(file)
with open('/home/tessj/wus_temp_trends/modmembers_prw_ssp245.json', 'r') as file:
        modmembers_prw_ssp245 = json.load(file)
        
del modmembers_hfls_ssp245['MCM-UA-1-0']
del modmembers_hfss_ssp245['MCM-UA-1-0']

In [7]:
modmembers_hfls_hist = {}
modmembers_hfss_hist = {}
modmembers_clt_hist = {}
modmembers_rsds_hist = {}
modmembers_rsus_hist = {}
modmembers_rsdt_hist = {}
modmembers_rsut_hist = {}

with open('/home/tessj/wus_temp_trends/modmembers_hfls_hist.json', 'r') as file:
        modmembers_hfls_hist = json.load(file)
with open('/home/tessj/wus_temp_trends/modmembers_hfss_hist.json', 'r') as file:
        modmembers_hfss_hist = json.load(file)
with open('/home/tessj/wus_temp_trends/modmembers_clt_hist.json', 'r') as file:
        modmembers_clt_hist = json.load(file)
with open('/home/tessj/wus_temp_trends/modmembers_rsds_hist.json', 'r') as file:
        modmembers_rsds_hist = json.load(file)
with open('/home/tessj/wus_temp_trends/modmembers_rsus_hist.json', 'r') as file:
        modmembers_rsus_hist = json.load(file)
with open('/home/tessj/wus_temp_trends/modmembers_rsdt_hist.json', 'r') as file:
        modmembers_rsdt_hist = json.load(file)
with open('/home/tessj/wus_temp_trends/modmembers_rsut_hist.json', 'r') as file:
        modmembers_rsut_hist = json.load(file)
with open('/home/tessj/wus_temp_trends/modmembers_rsut_hist.json', 'r') as file:
        modmembers_rsut_hist = json.load(file)
with open('/home/tessj/wus_temp_trends/modmembers_prw_hist.json', 'r') as file:
        modmembers_prw_hist = json.load(file)
        
del modmembers_hfls_hist['MCM-UA-1-0']
del modmembers_hfss_hist['MCM-UA-1-0']

In [8]:
del modmembers_huss_ssp245['MCM-UA-1-0']
del modmembers_huss_hist['MCM-UA-1-0']

In [9]:
del modmembers_hus300_ssp245['MCM-UA-1-0']
del modmembers_hus300_hist['MCM-UA-1-0']

In [10]:
del modmembers_rlut_hist['MCM-UA-1-0']
del modmembers_rlut_ssp245['MCM-UA-1-0']
del modmembers_rlut_hist['ICON-ESM-LR']

In [11]:
# list of ensemble members
modmembers_tas_ssp245 = {
'GFDL-CM4': ['r1i1p1f1'],
'GFDL-ESM4': ['r3i1p1f1', 'r2i1p1f1', 'r1i1p1f1'],
'IPSL-CM6A-LR': ['r1i1p1f1', 'r2i1p1f1', 'r10i1p1f1', 'r11i1p1f1', 'r5i1p1f1', 'r4i1p1f1', 'r6i1p1f1', 'r14i1p1f1', 'r3i1p1f1', 'r25i1p1f1', 'r22i1p1f1'],
'CNRM-CM6-1': ['r1i1p1f2', 'r3i1p1f2', 'r5i1p1f2', 'r4i1p1f2', 'r6i1p1f2', 'r2i1p1f2', 'r9i1p1f2', 'r7i1p1f2', 'r8i1p1f2', 'r10i1p1f2'],
'MRI-ESM2-0': ['r1i1p1f1', 'r3i3p1f1', 'r2i3p1f1', 'r5i3p1f1', 'r1i3p1f1', 'r4i3p1f1', 'r2i1p1f1', 'r3i1p1f1', 'r5i1p1f1', 'r4i1p1f1'],
'BCC-CSM2-MR': ['r1i1p1f1'],
'CNRM-ESM2-1': ['r1i1p1f2', 'r3i1p1f2', 'r4i1p1f2', 'r5i1p1f2', 'r2i1p1f2', 'r7i1p1f2', 'r8i1p1f2', 'r6i1p1f2', 'r10i1p1f2', 'r9i1p1f2'],
'CanESM5': ['r21i1p1f1', 'r20i1p2f1', 'r7i1p2f1', 'r8i1p2f1', 'r8i1p1f1', 'r6i1p1f1', 'r6i1p2f1', 'r7i1p1f1', 'r21i1p2f1', 'r9i1p2f1', 'r9i1p1f1', 'r3i1p2f1', 'r4i1p1f1', 'r3i1p1f1', 'r2i1p1f1', 'r2i1p2f1', 'r1i1p2f1', 'r1i1p1f1', 'r20i1p1f1', 'r23i1p2f1', 'r22i1p2f1', 'r23i1p1f1', 'r4i1p2f1', 'r5i1p1f1', 'r5i1p2f1', 'r24i1p1f1', 'r24i1p2f1', 'r25i1p1f1', 'r25i1p2f1', 'r22i1p1f1', 'r16i1p2f1', 'r13i1p2f1', 'r16i1p1f1', 'r14i1p1f1', 'r11i1p1f1', 'r18i1p1f1', 'r18i1p2f1', 'r17i1p1f1', 'r17i1p2f1', 'r10i1p1f1', 'r15i1p2f1', 'r12i1p1f1', 'r12i1p2f1', 'r10i1p2f1', 'r14i1p2f1', 'r15i1p1f1', 'r13i1p1f1', 'r11i1p2f1', 'r19i1p2f1', 'r19i1p1f1'],
'CanESM5-CanOE': ['r2i1p2f1', 'r3i1p2f1', 'r1i1p2f1'],
'UKESM1-0-LL': ['r3i1p1f2', 'r2i1p1f2', 'r4i1p1f2', 'r1i1p1f2', 'r8i1p1f2', 'r13i1p1f2', 'r5i1p1f2', 'r6i1p1f2', 'r11i1p1f2', 'r10i1p1f2', 'r9i1p1f2', 'r16i1p1f2', 'r19i1p1f2', 'r17i1p1f2', 'r18i1p1f2', 'r7i1p1f2', 'r12i1p1f2'],
'AWI-CM-1-1-MR': ['r1i1p1f1'],
'INM-CM4-8': ['r1i1p1f1'],
'INM-CM5-0': ['r1i1p1f1'],
'MIROC6': ['r1i1p1f1', 'r3i1p1f1', 'r2i1p1f1', 'r6i1p1f1', 'r4i1p1f1', 'r5i1p1f1', 'r11i1p1f1', 'r13i1p1f1', 'r20i1p1f1', 'r19i1p1f1', 'r10i1p1f1', 'r16i1p1f1', 'r18i1p1f1', 'r17i1p1f1', 'r15i1p1f1', 'r14i1p1f1', 'r12i1p1f1', 'r7i1p1f1', 'r9i1p1f1', 'r8i1p1f1', 'r47i1p1f1', 'r38i1p1f1', 'r49i1p1f1', 'r24i1p1f1', 'r23i1p1f1', 'r39i1p1f1', 'r25i1p1f1', 'r46i1p1f1', 'r45i1p1f1', 'r42i1p1f1', 'r43i1p1f1', 'r22i1p1f1', 'r37i1p1f1', 'r44i1p1f1', 'r28i1p1f1', 'r48i1p1f1', 'r21i1p1f1', 'r26i1p1f1', 'r34i1p1f1', 'r33i1p1f1', 'r36i1p1f1', 'r35i1p1f1', 'r32i1p1f1', 'r31i1p1f1', 'r30i1p1f1', 'r40i1p1f1', 'r41i1p1f1', 'r27i1p1f1', 'r50i1p1f1', 'r29i1p1f1'],
'CAMS-CSM1-0': ['r1i1p1f1', 'r2i1p1f1'],
'MPI-ESM1-2-LR': ['r4i1p1f1', 'r2i1p1f1', 'r10i1p1f1', 'r3i1p1f1', 'r5i1p1f1', 'r6i1p1f1', 'r7i1p1f1', 'r8i1p1f1', 'r1i1p1f1', 'r9i1p1f1'],
'MPI-ESM1-2-HR': ['r2i1p1f1', 'r1i1p1f1'],
'NESM3': ['r2i1p1f1', 'r1i1p1f1'],
'CESM2-WACCM': ['r1i1p1f1', 'r4i1p1f1', 'r5i1p1f1', 'r3i1p1f1', 'r2i1p1f1'],
'FGOALS-g3': ['r1i1p1f1', 'r3i1p1f1', 'r2i1p1f1', 'r4i1p1f1'],
'MIROC-ES2L': ['r1i1p1f2', 'r17i1p1f2', 'r25i1p1f2', 'r30i1p1f2', 'r8i1p1f2', 'r13i1p1f2', 'r12i1p1f2', 'r11i1p1f2', 'r5i1p1f2', 'r4i1p1f2', 'r16i1p1f2', 'r3i1p1f2', 'r15i1p1f2', 'r23i1p1f2', 'r2i1p1f2', 'r21i1p1f2', 'r27i1p1f2', 'r26i1p1f2', 'r18i1p1f2', 'r19i1p1f2', 'r6i1p1f2', 'r14i1p1f2', 'r22i1p1f2', 'r9i1p1f2', 'r28i1p1f2', 'r29i1p1f2', 'r20i1p1f2', 'r24i1p1f2', 'r7i1p1f2', 'r10i1p1f2'],
'HadGEM3-GC31-LL': ['r2i1p1f3', 'r4i1p1f3', 'r3i1p1f3', 'r1i1p1f3'],
'FGOALS-f3-L': ['r1i1p1f1'],
'NorESM2-LM': ['r1i1p1f1', 'r3i1p1f1', 'r2i1p1f1', 'r5i1p1f2', 'r9i1p1f2', 'r8i1p1f2', 'r2i1p1f2', 'r6i1p1f2', 'r7i1p1f2', 'r4i1p1f2', 'r3i1p1f2', 'r1i1p1f2', 'r10i1p1f2'],
'ACCESS-CM2': ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1', 'r4i1p1f1', 'r5i1p1f1'],
'NorESM2-MM': ['r1i1p1f1', 'r2i1p1f1'],
'CNRM-CM6-1-HR': ['r1i1p1f2'],
'KACE-1-0-G': ['r2i1p1f1', 'r1i1p1f1', 'r3i1p1f1'],
'FIO-ESM-2-0': ['r3i1p1f1', 'r2i1p1f1', 'r1i1p1f1'],
'GISS-E2-1-G': ['r5i1p5f2', 'r6i1p5f2', 'r7i1p1f2', 'r8i1p5f2', 'r6i1p1f2', 'r8i1p1f2', 'r9i1p1f2', 'r9i1p5f2', 'r7i1p5f2', 'r5i1p3f1', 'r4i1p5f2', 'r4i1p3f1', 'r4i1p5f1', 'r5i1p5f1', 'r3i1p5f2', 'r10i1p5f2', 'r10i1p1f2', 'r1i1p5f1', 'r1i1p3f1', 'r1i1p5f2', 'r3i1p5f1', 'r3i1p3f1', 'r2i1p3f1', 'r2i1p5f1', 'r2i1p5f2'],
'GISS-E2-1-H': ['r4i1p1f2', 'r5i1p1f2', 'r3i1p1f2', 'r1i1p1f2', 'r2i1p1f2', 'r4i1p3f1', 'r2i1p3f1', 'r3i1p3f1', 'r1i1p3f1', 'r5i1p3f1'],
'EC-Earth3-Veg': ['r6i1p1f1', 'r1i1p1f1', 'r2i1p1f1', 'r4i1p1f1', 'r3i1p1f1', 'r12i1p1f1', 'r14i1p1f1', 'r5i1p1f1'],
'EC-Earth3': ['r11i1p1f1', 'r15i1p1f1', 'r6i1p1f1', 'r13i1p1f1', 'r9i1p1f1', 'r1i1p1f1', 'r4i1p1f1', 'r15i1p1f2', 'r27i1p1f2', 'r29i1p1f2', 'r25i1p1f2', 'r19i1p1f2', 'r12i1p1f2', 'r17i1p1f2', 'r21i1p1f2', 'r23i1p1f2', 'r8i1p1f2', 'r1i1p1f2', 'r2i1p1f2', 'r30i1p1f2', 'r2i1p1f1', 'r7i1p1f1', 'r14i1p1f1', 'r10i1p1f1', 'r12i1p1f1', 'r16i1p1f1', 'r17i1p1f1', 'r18i1p1f1', 'r19i1p1f1', 'r7i1p1f2', 'r20i1p1f1', 'r21i1p1f1', 'r126i1p1f1', 'r137i1p1f1', 'r127i1p1f1', 'r131i1p1f1', 'r135i1p1f1', 'r136i1p1f1', 'r139i1p1f1', 'r140i1p1f1', 'r138i1p1f1', 'r129i1p1f1', 'r128i1p1f1', 'r124i1p1f1', 'r125i1p1f1', 'r118i1p1f1', 'r120i1p1f1', 'r117i1p1f1', 'r121i1p1f1', 'r123i1p1f1', 'r132i1p1f1', 'r133i1p1f1', 'r134i1p1f1', 'r130i1p1f1', 'r142i1p1f1', 'r122i1p1f1', 'r119i1p1f1', 'r104i1p1f1', 'r103i1p1f1', 'r105i1p1f1', 'r150i1p1f1', 'r110i1p1f1', 'r111i1p1f1', 'r106i1p1f1', 'r143i1p1f1', 'r114i1p1f1', 'r116i1p1f1', 'r109i1p1f1', 'r115i1p1f1', 'r102i1p1f1', 'r107i1p1f1', 'r113i1p1f1', 'r147i1p1f1', 'r141i1p1f1', 'r101i1p1f1', 'r112i1p1f1', 'r149i1p1f1', 'r145i1p1f1', 'r148i1p1f1', 'r108i1p1f1', 'r146i1p1f1', 'r144i1p1f1', 'r18i1p1f2', 'r16i1p1f2', 'r13i1p1f2', 'r20i1p1f2', 'r22i1p1f2', 'r24i1p1f2', 'r28i1p1f2', 'r4i1p1f2', 'r26i1p1f2', 'r10i1p1f2', 'r23i1p1f1', 'r6i1p1f2', 'r22i1p1f1', 'r24i1p1f1', 'r25i1p1f1'],
'CIESM': ['r1i1p1f1'],
'CESM2': ['r10i1p1f1', 'r4i1p1f1', 'r11i1p1f1'],
'CMCC-CM2-SR5': ['r1i1p1f1'],
'IITM-ESM': ['r1i1p1f1'],
'E3SM-1-1': ['r1i1p1f1', 'r3i1p1f1', 'r2i1p1f1', 'r10i1p1f1', 'r9i1p1f1', 'r8i1p1f1', 'r6i1p1f1', 'r5i1p1f1', 'r7i1p1f1', 'r4i1p1f1'],
'EC-Earth3-Veg-LR': ['r2i1p1f1', 'r1i1p1f1', 'r3i1p1f1'],
'TaiESM1': ['r1i1p1f1'],
'CAS-ESM2-0': ['r1i1p1f1', 'r3i1p1f1'],
'EC-Earth3-CC': ['r1i1p1f1'],
'CMCC-ESM2': ['r1i1p1f1'],
'KIOST-ESM': ['r1i1p1f1'],
'ACCESS-ESM1-5': ['r33i1p1f1', 'r31i1p1f1', 'r32i1p1f1', 'r38i1p1f1', 'r39i1p1f1', 'r40i1p1f1', 'r37i1p1f1', 'r34i1p1f1', 'r35i1p1f1', 'r36i1p1f1']
}

incomplete_tas_ssp245 = {
'CNRM-CM6-1': ['r7i1p1f2', 'r8i1p1f2', 'r10i1p1f2', 'r9i1p1f2'],
'HadGEM3-GC31-LL': ['r4i1p1f3', 'r3i1p1f3', 'r2i1p1f3'],
'E3SM-1-1': ['r1i1p1f1']
}

for k, v in modmembers_tas_ssp245.items():
    if k in incomplete_tas_ssp245.keys():
        modmembers_tas_ssp245[k] = [x for x in modmembers_tas_ssp245[k] if x not in incomplete_tas_ssp245[k]]

In [12]:
modmembers_pr_ssp245 = {
'GFDL-CM4': ['r1i1p1f1'],
'GFDL-ESM4': ['r2i1p1f1', 'r3i1p1f1', 'r1i1p1f1'],
'IPSL-CM6A-LR': ['r1i1p1f1', 'r2i1p1f1', 'r10i1p1f1', 'r5i1p1f1', 'r11i1p1f1', 'r4i1p1f1', 'r14i1p1f1', 'r6i1p1f1', 'r3i1p1f1', 'r22i1p1f1', 'r25i1p1f1'],
'CNRM-CM6-1': ['r1i1p1f2', 'r3i1p1f2', 'r6i1p1f2', 'r5i1p1f2', 'r4i1p1f2', 'r2i1p1f2', 'r8i1p1f2', 'r7i1p1f2', 'r10i1p1f2', 'r9i1p1f2'],
'MRI-ESM2-0': ['r1i1p1f1', 'r3i3p1f1', 'r2i3p1f1', 'r5i3p1f1', 'r4i3p1f1', 'r1i3p1f1', 'r2i1p1f1', 'r4i1p1f1', 'r3i1p1f1', 'r5i1p1f1'],
'BCC-CSM2-MR': ['r1i1p1f1'],
'CNRM-ESM2-1': ['r1i1p1f2', 'r4i1p1f2', 'r3i1p1f2', 'r5i1p1f2', 'r2i1p1f2', 'r7i1p1f2', 'r6i1p1f2', 'r9i1p1f2', 'r8i1p1f2', 'r10i1p1f2'],
'CanESM5': ['r20i1p2f1', 'r21i1p1f1', 'r7i1p2f1', 'r8i1p1f1', 'r8i1p2f1', 'r6i1p1f1', 'r6i1p2f1', 'r7i1p1f1', 'r21i1p2f1', 'r9i1p1f1', 'r9i1p2f1', 'r4i1p1f1', 'r3i1p2f1', 'r4i1p2f1', 'r2i1p1f1', 'r2i1p2f1', 'r3i1p1f1', 'r1i1p1f1', 'r1i1p2f1', 'r20i1p1f1', 'r23i1p2f1', 'r23i1p1f1', 'r22i1p2f1', 'r5i1p1f1', 'r5i1p2f1', 'r24i1p1f1', 'r24i1p2f1', 'r25i1p1f1', 'r25i1p2f1', 'r22i1p1f1', 'r17i1p1f1', 'r16i1p2f1', 'r13i1p2f1', 'r18i1p2f1', 'r16i1p1f1', 'r14i1p1f1', 'r18i1p1f1', 'r11i1p1f1', 'r17i1p2f1', 'r10i1p1f1', 'r15i1p2f1', 'r12i1p1f1', 'r12i1p2f1', 'r14i1p2f1', 'r15i1p1f1', 'r19i1p2f1', 'r11i1p2f1', 'r10i1p2f1', 'r13i1p1f1', 'r19i1p1f1'],
'CanESM5-CanOE': ['r3i1p2f1', 'r1i1p2f1', 'r2i1p2f1'],
'UKESM1-0-LL': ['r2i1p1f2', 'r3i1p1f2', 'r4i1p1f2', 'r1i1p1f2', 'r8i1p1f2', 'r13i1p1f2', 'r6i1p1f2', 'r5i1p1f2', 'r9i1p1f2', 'r10i1p1f2', 'r16i1p1f2', 'r17i1p1f2', 'r11i1p1f2', 'r19i1p1f2', 'r18i1p1f2', 'r7i1p1f2', 'r12i1p1f2'],
'AWI-CM-1-1-MR': ['r1i1p1f1'],
'INM-CM4-8': ['r1i1p1f1'],
'INM-CM5-0': ['r1i1p1f1'],
'MIROC6': ['r1i1p1f1', 'r3i1p1f1', 'r2i1p1f1', 'r6i1p1f1', 'r50i1p1f1', 'r4i1p1f1', 'r5i1p1f1', 'r11i1p1f1', 'r13i1p1f1', 'r20i1p1f1', 'r19i1p1f1', 'r10i1p1f1', 'r17i1p1f1', 'r16i1p1f1', 'r18i1p1f1', 'r15i1p1f1', 'r12i1p1f1', 'r22i1p1f1', 'r14i1p1f1', 'r7i1p1f1', 'r9i1p1f1', 'r8i1p1f1', 'r41i1p1f1', 'r28i1p1f1', 'r25i1p1f1', 'r26i1p1f1', 'r24i1p1f1', 'r23i1p1f1', 'r48i1p1f1', 'r46i1p1f1', 'r49i1p1f1', 'r47i1p1f1', 'r45i1p1f1', 'r44i1p1f1', 'r39i1p1f1', 'r35i1p1f1', 'r34i1p1f1', 'r33i1p1f1', 'r29i1p1f1', 'r30i1p1f1', 'r21i1p1f1', 'r32i1p1p1f1', 'r31i1p1f1', 'r37i1p1f1', 'r40i1p1f1', 'r27i1p1f1', 'r42i1p1f1', 'r38i1p1f1', 'r43i1p1f1', 'r36i1p1f1'],
'CAMS-CSM1-0': ['r1i1p1f1', 'r2i1p1f1'],
'MPI-ESM1-2-LR': ['r4i1p1f1', 'r2i1p1f1', 'r1i1p1f1', 'r10i1p1f1', 'r5i1p1f1', 'r3i1p1f1', 'r6i1p1f1', 'r7i1p1f1', 'r8i1p1f1', 'r9i1p1f1'],
'MPI-ESM1-2-HR': ['r2i1p1f1', 'r1i1p1f1'],
'NESM3': ['r2i1p1f1', 'r1i1p1f1'],
'CESM2-WACCM': ['r1i1p1f1', 'r5i1p1f1', 'r4i1p1f1', 'r3i1p1f1', 'r2i1p1f1'],
'FGOALS-g3': ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1', 'r4i1p1f1'],
'MIROC-ES2L': ['r1i1p1f2', 'r8i1p1f2', 'r13i1p1f2', 'r17i1p1f2', 'r12i1p1f2', 'r11i1p1f2', 'r6i1p1f2', 'r25i1p1f2', 'r5i1p1f2', 'r15i1p1f2', 'r16i1p1f2', 'r4i1p1f2', 'r3i1p1f2', 'r30i1p1f2', 'r23i1p1f2', 'r21i1p1f2', 'r27i1p1f2', 'r19i1p1f2', 'r26i1p1f2', 'r24i1p1f2', 'r18i1p1f2', 'r14i1p1f2', 'r22i1p1f2', 'r9i1p1f2', 'r2i1p1f2', 'r28i1p1f2', 'r29i1p1f2', 'r20i1p1f2', 'r10i1p1f2', 'r7i1p1f2'],
'HadGEM3-GC31-LL': ['r2i1p1f3', 'r4i1p1f3', 'r1i1p1f3', 'r3i1p1f3'],
'FGOALS-f3-L': ['r1i1p1f1'],
'NorESM2-LM': ['r3i1p1f1', 'r1i1p1f1', 'r2i1p1f1', 'r9i1p1f2', 'r8i1p1f2', 'r5i1p1f2', 'r2i1p1f2', 'r6i1p1f2', 'r7i1p1f2', 'r4i1p1f2', 'r3i1p1f2', 'r1i1p1f2', 'r10i1p1f2'],
'ACCESS-CM2': ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1', 'r4i1p1f1', 'r5i1p1f1'],
'NorESM2-MM': ['r1i1p1f1', 'r2i1p1f1'],
'KACE-1-0-G': ['r3i1p1f1', 'r2i1p1f1', 'r1i1p1f1'],
'CNRM-CM6-1-HR': ['r1i1p1f2'],
'FIO-ESM-2-0': ['r1i1p1f1', 'r3i1p1f1', 'r2i1p1f1'],
'GISS-E2-1-G': ['r7i1p1f2', 'r6i1p5f2', 'r8i1p5f2', 'r7i1p5f2', 'r6i1p1f2', 'r9i1p5f2', 'r9i1p1f2', 'r8i1p1f2', 'r4i1p3f1', 'r5i1p5f2', 'r5i1p3f1', 'r4i1p5f2', 'r4i1p5f1', 'r5i1p5f1', 'r10i1p1f2', 'r10i1p5f2', 'r1i1p5f1', 'r1i1p3f1', 'r1i1p5f2', 'r3i1p5f2', 'r3i1p5f1', 'r3i1p3f1', 'r2i1p3f1', 'r2i1p5f1', 'r2i1p5f2'],
'GISS-E2-1-H': ['r1i1p1f2', 'r5i1p1f2', 'r2i1p1f2', 'r3i1p1f2', 'r4i1p1f2', 'r2i1p3f1', 'r4i1p3f1', 'r3i1p3f1', 'r1i1p3f1', 'r5i1p3f1'],
'EC-Earth3-Veg': ['r6i1p1f1', 'r1i1p1f1', 'r2i1p1f1', 'r4i1p1f1', 'r3i1p1f1', 'r14i1p1f1', 'r12i1p1f1', 'r5i1p1f1'],
'EC-Earth3': ['r11i1p1f1', 'r6i1p1f1', 'r15i1p1f1', 'r9i1p1f1', 'r13i1p1f1', 'r1i1p1f1', 'r4i1p1f1', 'r15i1p1f2', 'r12i1p1f2', 'r19i1p1f2', 'r25i1p1f2', 'r27i1p1f2', 'r29i1p1f2', 'r17i1p1f2', 'r21i1p1f2', 'r23i1p1f2', 'r8i1p1f2', 'r1i1p1f2', 'r30i1p1f2', 'r2i1p1f2', 'r2i1p1f1', 'r7i1p1f1', 'r14i1p1f1', 'r10i1p1f1', 'r12i1p1f1', 'r16i1p1f1', 'r17i1p1f1', 'r18i1p1f1', 'r19i1p1f1', 'r7i1p1f2', 'r20i1p1f1', 'r21i1p1f1', 'r138i1p1f1', 'r117i1p1f1', 'r127i1p1f1', 'r126i1p1f1', 'r137i1p1f1', 'r135i1p1f1', 'r136i1p1f1', 'r140i1p1f1', 'r129i1p1f1', 'r128i1p1f1', 'r141i1p1f1', 'r124i1p1f1', 'r139i1p1f1', 'r125i1p1f1', 'r121i1p1f1', 'r120i1p1f1', 'r142i1p1f1', 'r118i1p1f1', 'r123i1p1f1', 'r122i1p1f1', 'r134i1p1f1', 'r132i1p1f1', 'r131i1p1f1', 'r130i1p1f1', 'r133i1p1f1', 'r119i1p1f1', 'r104i1p1f1', 'r105i1p1f1', 'r147i1p1f1', 'r150i1p1f1', 'r101i1p1f1', 'r111i1p1f1', 'r110i1p1f1', 'r114i1p1f1', 'r106i1p1f1', 'r143i1p1f1', 'r116i1p1f1', 'r115i1p1f1', 'r109i1p1f1', 'r102i1p1f1', 'r149i1p1f1', 'r107i1p1f1', 'r112i1p1f1', 'r144i1p1f1', 'r146i1p1f1', 'r145i1p1f1', 'r148i1p1f1', 'r108i1p1f1', 'r113i1p1f1', 'r103i1p1f1', 'r18i1p1f2', 'r10i1p1f2', 'r20i1p1f2', 'r13i1p1f2', 'r22i1p1f2', 'r26i1p1f2', 'r4i1p1f2', 'r24i1p1f2', 'r28i1p1f2', 'r16i1p1f2', 'r23i1p1f1', 'r22i1p1f1', 'r24i1p1f1', 'r25i1p1f1'],
'CIESM': ['r1i1p1f1'],
'CESM2': ['r10i1p1f1', 'r4i1p1f1', 'r11i1p1f1'],
'CMCC-CM2-SR5': ['r1i1p1f1'],
'IITM-ESM': ['r1i1p1f1'],
'E3SM-1-1': ['r1i1p1f1', 'r3i1p1f1', 'r10i1p1f1', 'r2i1p1f1', 'r5i1p1f1', 'r4i1p1f1', 'r9i1p1f1', 'r8i1p1f1', 'r7i1p1f1', 'r6i1p1f1'],
'EC-Earth3-Veg-LR': ['r2i1p1f1', 'r1i1p1f1', 'r3i1p1f1'],
'TaiESM1': ['r1i1p1f1'],
'CAS-ESM2-0': ['r3i1p1f1', 'r1i1p1f1'],
'EC-Earth3-CC': ['r1i1p1f1'],
'CMCC-ESM2': ['r1i1p1f1'],
'ACCESS-ESM1-5': ['r33i1p1f1', 'r31i1p1f1', 'r32i1p1f1', 'r35i1p1f1', 'r38i1p1f1', 'r39i1p1f1', 'r40i1p1f1', 'r34i1p1f1', 'r37i1p1f1', 'r36i1p1f1'],
'KIOST-ESM': ['r1i1p1f1']
}

incomplete_pr_ssp245 = {
'CNRM-CM6-1': ['r7i1p1f2', 'r8i1p1f2', 'r10i1p1f2', 'r9i1p1f2'],
'HadGEM3-GC31-LL': ['r3i1p1f3', 'r4i1p1f3', 'r2i1p1f3'],
'E3SM-1-1': ['r1i1p1f1']
}

for k, v in modmembers_pr_ssp245.items():
    if k in incomplete_pr_ssp245.keys():
        modmembers_pr_ssp245[k] = [x for x in modmembers_pr_ssp245[k] if x not in incomplete_pr_ssp245[k]]

In [13]:
modmembers_zg700_hist = {'GFDL-ESM4': ['r3i1p1f1', 'r2i1p1f1', 'r1i1p1f1'],
 'GFDL-CM4': ['r1i1p1f1'],
 'IPSL-CM6A-LR': ['r2i1p1f1', 'r30i1p1f1', 'r8i1p1f1', 'r29i1p1f1', 'r3i1p1f1', 'r6i1p1f1', 'r27i1p1f1', 'r7i1p1f1', 'r26i1p1f1', 'r20i1p1f1', 'r25i1p1f1', 'r23i1p1f1', 'r9i1p1f1', 'r24i1p1f1', 'r22i1p1f1', 'r31i1p1f1', 'r21i1p1f1', 'r19i1p1f1', 'r10i1p1f1', 'r18i1p1f1', 'r12i1p1f1', 'r11i1p1f1', 'r17i1p1f1', 'r16i1p1f1', 'r1i1p1f1', 'r4i1p1f1', 'r5i1p1f1', 'r28i1p1f1', 'r14i1p1f1', 'r15i1p1f1', 'r13i1p1f1', 'r32i1p1f1'],
 'GISS-E2-1-G': ['r2i1p1f1', 'r1i1p1f1', 'r5i1p1f1', 'r4i1p1f1', 'r3i1p1f1', 'r7i1p1f1', 'r6i1p1f1', 'r9i1p1f1', 'r10i1p1f1', 'r8i1p1f1', 'r3i1p3f1', 'r1i1p3f1', 'r2i1p3f1', 'r10i1p3f1', 'r9i1p3f1', 'r8i1p3f1', 'r5i1p3f1', 'r4i1p3f1', 'r6i1p3f1', 'r101i1p1f1', 'r102i1p1f1', 'r3i1p1f3', 'r3i1p1f2', 'r4i1p1f2', 'r5i1p1f2', 'r10i1p1f2', 'r11i1p1f2', 'r5i1p1f3', 'r6i1p1f2', 'r9i1p1f2', 'r8i1p1f2', 'r7i1p1f2', 'r2i1p1f3', 'r1i1p1f2', 'r4i1p1f3', 'r2i1p1f2', 'r1i1p1f3', 'r6i1p5f1', 'r3i1p5f1', 'r7i1p5f1', 'r8i1p5f1', 'r9i1p5f1', 'r2i1p5f1', 'r4i1p5f1', 'r1i1p5f1', 'r10i1p5f1'],
 'CNRM-CM6-1': ['r1i1p1f2', 'r2i1p1f2', 'r7i1p1f2', 'r9i1p1f2', 'r6i1p1f2', 'r5i1p1f2', 'r4i1p1f2', 'r8i1p1f2', 'r10i1p1f2', 'r3i1p1f2', 'r14i1p1f2', 'r18i1p1f2', 'r21i1p1f2', 'r23i1p1f2', 'r11i1p1f2', 'r26i1p1f2', 'r22i1p1f2', 'r15i1p1f2', 'r16i1p1f2', 'r13i1p1f2', 'r17i1p1f2', 'r30i1p1f2', 'r19i1p1f2', 'r27i1p1f2', 'r25i1p1f2', 'r28i1p1f2', 'r12i1p1f2', 'r24i1p1f2', 'r29i1p1f2'],
 'BCC-CSM2-MR': ['r2i1p1f1', 'r3i1p1f1', 'r1i1p1f1'],
 'CNRM-ESM2-1': ['r1i1p1f2', 'r3i1p1f2', 'r5i1p1f2', 'r2i1p1f2', 'r4i1p1f2', 'r10i1p1f2', 'r9i1p1f2', 'r8i1p1f2', 'r7i1p1f2'],
 'BCC-ESM1': ['r2i1p1f1', 'r3i1p1f1', 'r1i1p1f1'],
 'AWI-CM-1-1-MR': ['r3i1p1f1', 'r5i1p1f1', 'r2i1p1f1', 'r1i1p1f1', 'r4i1p1f1'],
 'CESM2-WACCM': ['r2i1p1f1', 'r3i1p1f1', 'r1i1p1f1'],
 'CESM2': ['r1i1p1f1', 'r4i1p1f1', 'r5i1p1f1', 'r3i1p1f1', 'r2i1p1f1', 'r6i1p1f1', 'r8i1p1f1', 'r7i1p1f1', 'r9i1p1f1', 'r10i1p1f1', 'r11i1p1f1'],
 'MRI-ESM2-0': ['r5i1p1f1', 'r4i1p1f1', 'r3i1p1f1', 'r2i1p1f1', 'r1i1p1f1', 'r1i2p1f1'],
 'MIROC6': ['r7i1p1f1', 'r5i1p1f1', 'r1i1p1f1', 'r10i1p1f1', 'r9i1p1f1', 'r8i1p1f1', 'r2i1p1f1', 'r6i1p1f1', 'r3i1p1f1', 'r4i1p1f1', 'r46i1p1f1', 'r47i1p1f1', 'r45i1p1f1', 'r30i1p1f1', 'r40i1p1f1', 'r41i1p1f1', 'r39i1p1f1', 'r50i1p1f1', 'r48i1p1f1', 'r49i1p1f1', 'r37i1p1f1', 'r38i1p1f1', 'r36i1p1f1', 'r35i1p1f1', 'r34i1p1f1', 'r33i1p1f1', 'r27i1p1f1', 'r31i1p1f1', 'r32i1p1f1', 'r29i1p1f1', 'r28i1p1f1', 'r14i1p1f1', 'r13i1p1f1', 'r24i1p1f1', 'r23i1p1f1', 'r11i1p1f1', 'r15i1p1f1', 'r25i1p1f1', 'r21i1p1f1', 'r22i1p1f1', 'r26i1p1f1', 'r44i1p1f1', 'r42i1p1f1', 'r17i1p1f1', 'r43i1p1f1', 'r18i1p1f1', 'r12i1p1f1', 'r20i1p1f1', 'r19i1p1f1', 'r16i1p1f1'],
 'SAM0-UNICON': ['r1i1p1f1'],
 'GISS-E2-1-H': ['r1i1p1f1', 'r6i1p1f1', 'r2i1p1f1', 'r3i1p1f1', 'r5i1p1f1', 'r10i1p1f1', 'r8i1p1f1', 'r4i1p1f1', 'r9i1p1f1', 'r7i1p1f1', 'r3i1p5f1', 'r2i1p5f1', 'r1i1p5f1', 'r4i1p5f1', 'r5i1p5f1', 'r5i1p1f2', 'r4i1p1f2', 'r3i1p1f2', 'r2i1p1f2', 'r1i1p1f2', 'r5i1p3f1', 'r4i1p3f1', 'r2i1p3f1', 'r1i1p3f1', 'r3i1p3f1'],
 'UKESM1-0-LL': ['r1i1p1f2', 'r3i1p1f2', 'r8i1p1f2', 'r2i1p1f2', 'r4i1p1f2', 'r5i1p1f3', 'r7i1p1f3', 'r9i1p1f2', 'r6i1p1f3', 'r17i1p1f2', 'r19i1p1f2', 'r18i1p1f2', 'r11i1p1f2', 'r16i1p1f2', 'r12i1p1f2', 'r10i1p1f2', 'r13i1p1f2', 'r14i1p1f2'],
 'CanESM5': ['r11i1p1f1', 'r10i1p2f1', 'r11i1p2f1', 'r13i1p1f1', 'r10i1p1f1', 'r13i1p2f1', 'r12i1p1f1', 'r12i1p2f1', 'r25i1p2f1', 'r9i1p2f1', 'r8i1p2f1', 'r9i1p1f1', 'r4i1p1f1', 'r40i1p2f1', 'r4i1p2f1', 'r5i1p2f1', 'r7i1p1f1', 'r7i1p2f1', 'r8i1p1f1', 'r5i1p1f1', 'r1i1p2f1', 'r19i1p1f1', 'r20i1p1f1', 'r21i1p1f1', 'r20i1p2f1', 'r15i1p2f1', 'r21i1p2f1', 'r1i1p1f1', 'r19i1p2f1', 'r22i1p1f1', 'r24i1p1f1', 'r38i1p2f1', 'r3i1p1f1', 'r39i1p2f1', 'r24i1p2f1', 'r3i1p2f1', 'r36i1p2f1', 'r37i1p2f1', 'r29i1p2f1', 'r26i1p2f1', 'r27i1p2f1', 'r2i1p1f1', 'r25i1p1f1', 'r22i1p2f1', 'r28i1p2f1', 'r23i1p2f1', 'r23i1p1f1', 'r18i1p1f1', 'r17i1p1f1', 'r18i1p2f1', 'r16i1p1f1', 'r15i1p1f1', 'r14i1p2f1', 'r14i1p1f1', 'r2i1p2f1', 'r33i1p2f1', 'r35i1p2f1', 'r34i1p2f1', 'r30i1p2f1', 'r17i1p2f1', 'r16i1p2f1', 'r31i1p2f1', 'r32i1p2f1', 'r6i1p2f1', 'r6i1p1f1'],
 'CanESM5-CanOE': ['r1i1p2f1', 'r3i1p2f1', 'r2i1p2f1'],
 'INM-CM4-8': ['r1i1p1f1'],
 'INM-CM5-0': ['r1i1p1f1', 'r3i1p1f1', 'r2i1p1f1', 'r4i1p1f1'],
 'HadGEM3-GC31-LL': ['r4i1p1f3', 'r2i1p1f3', 'r1i1p1f3', 'r3i1p1f3', 'r5i1p1f3'],
 'MPI-ESM-1-2-HAM': ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1'],
 'NESM3': ['r2i1p1f1', 'r5i1p1f1', 'r3i1p1f1', 'r4i1p1f1', 'r1i1p1f1'],
 'CAMS-CSM1-0': ['r1i1p1f1', 'r2i1p1f1', 'r1i1p1f2'],
 'MPI-ESM1-2-LR': ['r1i1p1f1', 'r10i1p1f1', 'r3i1p1f1', 'r2i1p1f1', 'r5i1p1f1', 'r6i1p1f1', 'r7i1p1f1', 'r8i1p1f1', 'r9i1p1f1'],
 'MPI-ESM1-2-HR': ['r2i1p1f1', 'r3i1p1f1', 'r10i1p1f1', 'r1i1p1f1', 'r9i1p1f1', 'r8i1p1f1', 'r5i1p1f1', 'r4i1p1f1', 'r7i1p1f1', 'r6i1p1f1'],
 'E3SM-1-0': ['r2i1p1f1', 'r5i1p1f1', 'r4i1p1f1', 'r3i1p1f1', 'r1i1p1f1'],
 'GISS-E2-1-G-CC': ['r1i1p1f1'],
 'NorESM2-LM': ['r1i1p1f1', 'r3i1p1f1', 'r2i1p1f1'],
 'FGOALS-g3': ['r1i1p1f1', 'r3i1p1f1', 'r2i1p1f1', 'r4i1p1f1', 'r6i1p1f1', 'r5i1p1f1'],
 'MIROC-ES2L': ['r2i1p1f2', 'r1i1p1f2', 'r3i1p1f2', 'r6i1p1f2', 'r9i1p1f2', 'r10i1p1f2', 'r8i1p1f2', 'r7i1p1f2', 'r4i1p1f2', 'r5i1p1f2', 'r1i1000p1f2', 'r23i1p1f2', 'r22i1p1f2', 'r11i1p1f2', 'r15i1p1f2', 'r17i1p1f2', 'r18i1p1f2', 'r30i1p1f2', 'r19i1p1f2', 'r26i1p1f2', 'r28i1p1f2', 'r21i1p1f2', 'r24i1p1f2', 'r25i1p1f2'],
 'NorCPM1': ['r10i1p1f1', 'r23i1p1f1', 'r29i1p1f1', 'r16i1p1f1', 'r17i1p1f1', 'r4i1p1f1', 'r30i1p1f1', 'r22i1p1f1', 'r25i1p1f1', 'r26i1p1f1', 'r28i1p1f1', 'r18i1p1f1', 'r27i1p1f1', 'r24i1p1f1', 'r19i1p1f1', 'r20i1p1f1', 'r2i1p1f1', 'r14i1p1f1', 'r15i1p1f1', 'r21i1p1f1', 'r1i1p1f1', 'r11i1p1f1', 'r12i1p1f1', 'r13i1p1f1', 'r9i1p1f1', 'r7i1p1f1', 'r6i1p1f1', 'r3i1p1f1', 'r5i1p1f1', 'r8i1p1f1'],
 'FGOALS-f3-L': ['r3i1p1f1', 'r1i1p1f1', 'r2i1p1f1'],
 'CNRM-CM6-1-HR': ['r1i1p1f2'],
 'KACE-1-0-G': ['r3i1p1f1', 'r2i1p1f1', 'r1i1p1f1'],
 'KIOST-ESM': ['r1i1p1f1'],
 'ACCESS-CM2': ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1'],
 'NorESM2-MM': ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1'],
 'ACCESS-ESM1-5': ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1', 'r9i1p1f1', 'r4i1p1f1', 'r6i1p1f1', 'r8i1p1f1', 'r7i1p1f1', 'r5i1p1f1', 'r10i1p1f1', 'r19i1p1f1', 'r18i1p1f1', 'r17i1p1f1', 'r20i1p1f1', 'r16i1p1f1', 'r14i1p1f1', 'r13i1p1f1', 'r15i1p1f1', 'r11i1p1f1', 'r12i1p1f1', 'r29i1p1f1', 'r28i1p1f1', 'r30i1p1f1', 'r25i1p1f1', 'r26i1p1f1', 'r24i1p1f1', 'r23i1p1f1', 'r27i1p1f1', 'r21i1p1f1', 'r22i1p1f1'],
 'CESM2-FV2': ['r1i1p1f1', 'r3i1p1f1', 'r2i1p1f1'],
 'CESM2-WACCM-FV2': ['r1i1p1f1', 'r3i1p1f1', 'r2i1p1f1'],
 'FIO-ESM-2-0': ['r3i1p1f1', 'r1i1p1f1', 'r2i1p1f1'],
 'HadGEM3-GC31-MM': ['r1i1p1f3', 'r2i1p1f3', 'r3i1p1f3'],
 'E3SM-1-1': ['r1i1p1f1'],
 'IITM-ESM': ['r1i1p1f1'],
 'EC-Earth3-Veg': ['r6i1p1f1', 'r1i1p1f1', 'r3i1p1f1', 'r2i1p1f1', 'r5i1p1f1', 'r4i1p1f1', 'r12i1p1f1', 'r14i1p1f1'],
 'EC-Earth3': ['r9i1p1f1', 'r11i1p1f1', 'r13i1p1f1', 'r6i1p1f1', 'r15i1p1f1', 'r22i1p1f1', 'r25i1p1f1', 'r23i1p1f1', 'r1i1p1f1', 'r12i1p1f1', 'r19i1p1f1', 'r24i1p1f1', 'r119i1p1f1', 'r122i1p1f1', 'r117i1p1f1', 'r118i1p1f1', 'r127i1p1f1', 'r126i1p1f1', 'r125i1p1f1', 'r120i1p1f1', 'r121i1p1f1', 'r149i1p1f1', 'r148i1p1f1', 'r106i1p1f1', 'r105i1p1f1', 'r101i1p1f1', 'r150i1p1f1', 'r104i1p1f1', 'r102i1p1f1', 'r103i1p1f1', 'r129i1p1f1', 'r136i1p1f1', 'r137i1p1f1', 'r131i1p1f1', 'r143i1p1f1', 'r145i1p1f1', 'r128i1p1f1', 'r130i1p1f1', 'r132i1p1f1', 'r133i1p1f1', 'r134i1p1f1', 'r135i1p1f1', 'r109i1p1f1', 'r108i1p1f1', 'r107i1p1f1', 'r116i1p1f1', 'r115i1p1f1', 'r124i1p1f1', 'r123i1p1f1', 'r111i1p1f1', 'r110i1p1f1', 'r139i1p1f1', 'r138i1p1f1', 'r140i1p1f1', 'r147i1p1f1', 'r144i1p1f1', 'r146i1p1f1', 'r113i1p1f1', 'r114i1p1f1', 'r141i1p1f1', 'r112i1p1f1', 'r142i1p1f1', 'r4i1p1f1', 'r2i1p1f1', 'r7i1p1f1', 'r14i1p1f1', 'r10i1p1f1', 'r16i1p1f1', 'r17i1p1f1', 'r18i1p1f1', 'r21i1p1f1'],
 'AWI-ESM-1-1-LR': ['r1i1p1f1'],
 'EC-Earth3-Veg-LR': ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1'],
 'CIESM': ['r2i1p1f1', 'r1i1p1f1', 'r3i1p1f1'],
 'CAS-ESM2-0': ['r4i1p1f1', 'r3i1p1f1', 'r1i1p1f1'],
 'CMCC-CM2-SR5': ['r1i1p1f1'],
 'TaiESM1': ['r1i1p1f1', 'r2i1p1f1'],
 'EC-Earth3-AerChem': ['r1i1p1f1', 'r4i1p1f1'],
 'E3SM-1-1-ECA': ['r1i1p1f1'],
 'CMCC-CM2-HR4': ['r1i1p1f1'],
 'EC-Earth3-CC': ['r1i1p1f1'],
 'CMCC-ESM2': ['r1i1p1f1'],
 'MIROC-ES2H': ['r1i1p2f2', 'r1i1p1f2', 'r1i1p3f2'],
 'IPSL-CM6A-LR-INCA': ['r1i1p1f1']}

incomplete_zg700_hist = {'EC-Earth3': ['r106i1p1f1', 'r114i1p1f1', 'r116i1p1f1', 'r112i1p1f1', 'r119i1p1f1', 'r110i1p1f1', 'r123i1p1f1', 'r120i1p1f1', 'r104i1p1f1', 'r115i1p1f1', 'r122i1p1f1', 'r103i1p1f1', 'r101i1p1f1', 'r121i1p1f1', 'r109i1p1f1', 'r102i1p1f1', 'r117i1p1f1', 'r118i1p1f1', 'r108i1p1f1', 'r113i1p1f1', 'r107i1p1f1', 'r111i1p1f1', 'r105i1p1f1', 'r124i1p1f1']}

for k, v in modmembers_zg700_hist.items():
    if k in incomplete_zg700_hist.keys():
        modmembers_zg700_hist[k] = [x for x in modmembers_zg700_hist[k] if x not in incomplete_zg700_hist[k]]


In [14]:
modmembers_zg700_ssp245 = {
    'GFDL-CM4': ['r1i1p1f1'],
    'GFDL-ESM4': ['r3i1p1f1', 'r2i1p1f1', 'r1i1p1f1'],
    'IPSL-CM6A-LR': ['r1i1p1f1', 'r2i1p1f1', 'r10i1p1f1', 'r11i1p1f1', 'r5i1p1f1', 'r4i1p1f1', 'r6i1p1f1', 'r14i1p1f1', 'r3i1p1f1', 'r25i1p1f1', 'r22i1p1f1'],
    'CNRM-CM6-1': ['r1i1p1f2', 'r3i1p1f2', 'r6i1p1f2', 'r4i1p1f2', 'r5i1p1f2', 'r2i1p1f2', 'r7i1p1f2', 'r8i1p1f2', 'r10i1p1f2', 'r9i1p1f2'],
    'MRI-ESM2-0': ['r3i1p1f1', 'r2i1p1f1', 'r1i1p1f1', 'r4i1p1f1', 'r5i1p1f1', 'r3i3p1f1', 'r2i3p1f1', 'r1i3p1f1', 'r4i3p1f1', 'r5i3p1f1'],
    'BCC-CSM2-MR': ['r1i1p1f1'],
    'CNRM-ESM2-1': ['r1i1p1f2', 'r4i1p1f2', 'r3i1p1f2', 'r5i1p1f2', 'r2i1p1f2', 'r7i1p1f2', 'r6i1p1f2', 'r8i1p1f2', 'r10i1p1f2', 'r9i1p1f2'],
    'CanESM5': ['r20i1p2f1', 'r7i1p2f1', 'r8i1p2f1', 'r8i1p1f1', 'r6i1p2f1', 'r6i1p1f1', 'r7i1p1f1', 'r21i1p2f1', 'r9i1p2f1', 'r9i1p1f1', 'r3i1p2f1', 'r4i1p1f1', 'r5i1p2f1', 'r2i1p1f1', 'r3i1p1f1', 'r2i1p2f1', 'r1i1p2f1', 'r1i1p1f1', 'r20i1p1f1', 'r21i1p1f1', 'r22i1p2f1', 'r23i1p1f1', 'r4i1p2f1', 'r5i1p1f1', 'r24i1p1f1', 'r24i1p2f1', 'r25i1p1f1', 'r23i1p2f1', 'r25i1p2f1', 'r22i1p1f1', 'r13i1p2f1', 'r16i1p2f1', 'r18i1p1f1', 'r16i1p1f1', 'r14i1p1f1', 'r11i1p1f1', 'r18i1p2f1', 'r17i1p1f1', 'r17i1p2f1', 'r10i1p1f1', 'r15i1p2f1', 'r12i1p1f1', 'r12i1p2f1', 'r10i1p2f1', 'r14i1p2f1', 'r15i1p1f1', 'r13i1p1f1', 'r19i1p1f1', 'r11i1p2f1', 'r19i1p2f1'],
    'CanESM5-CanOE': ['r2i1p2f1', 'r3i1p2f1', 'r1i1p2f1'],
    'UKESM1-0-LL': ['r3i1p1f2', 'r2i1p1f2', 'r4i1p1f2', 'r1i1p1f2', 'r8i1p1f2', 'r13i1p1f2', 'r6i1p1f2', 'r7i1p1f2', 'r9i1p1f2', 'r10i1p1f2', 'r11i1p1f2', 'r18i1p1f2', 'r16i1p1f2', 'r5i1p1f2', 'r17i1p1f2', 'r12i1p1f2', 'r19i1p1f2'],
    'AWI-CM-1-1-MR': ['r1i1p1f1'],
    'INM-CM4-8': ['r1i1p1f1'],
    'INM-CM5-0': ['r1i1p1f1'],
    'MIROC6': ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1', 'r29i1p1f1', 'r30i1p1f1', 'r32i1p1f1', 'r31i1p1f1', 'r34i1p1f1', 'r24i1p1f1', 'r23i1p1f1', 'r35i1p1f1', 'r33i1p1f1', 'r36i1p1f1', 'r40i1p1f1', 'r39i1p1f1', 'r37i1p1f1', 'r4i1p1f1', 'r50i1p1f1', 'r41i1p1f1', 'r42i1p1f1', 'r38i1p1f1', 'r44i1p1f1', 'r43i1p1f1', 'r45i1p1f1', 'r28i1p1f1', 'r26i1p1f1', 'r25i1p1f1', 'r27i1p1f1', 'r47i1p1f1', 'r6i1p1f1', 'r5i1p1f1', 'r48i1p1f1', 'r46i1p1f1', 'r49i1p1f1', 'r11i1p1f1', 'r13i1p1f1', 'r19i1p1f1', 'r20i1p1f1', 'r16i1p1f1', 'r21i1p1f1', 'r15i1p1f1', 'r18i1p1f1', 'r17i1p1f1', 'r14i1p1f1', 'r12i1p1f1', 'r22i1p1f1', 'r9i1p1f1', 'r7i1p1f1', 'r8i1p1f1', 'r10i1p1f1'],
    'CAMS-CSM1-0': ['r1i1p1f1', 'r2i1p1f1'],
    'MPI-ESM1-2-LR': ['r4i1p1f1', 'r2i1p1f1', 'r10i1p1f1', 'r3i1p1f1', 'r5i1p1f1', 'r6i1p1f1', 'r7i1p1f1', 'r8i1p1f1', 'r9i1p1f1', 'r1i1p1f1'],
    'MPI-ESM1-2-HR': ['r2i1p1f1', 'r1i1p1f1'],
    'NESM3': ['r2i1p1f1', 'r1i1p1f1'],
    'CESM2-WACCM': ['r1i1p1f1', 'r4i1p1f1', 'r5i1p1f1', 'r3i1p1f1', 'r2i1p1f1'],
    'FGOALS-g3': ['r1i1p1f1', 'r3i1p1f1', 'r2i1p1f1', 'r4i1p1f1'],
    'MIROC-ES2L': ['r1i1p1f2', 'r30i1p1f2', 'r8i1p1f2', 'r9i1p1f2', 'r7i1p1f2', 'r13i1p1f2', 'r12i1p1f2', 'r22i1p1f2', 'r4i1p1f2', 'r3i1p1f2', 'r15i1p1f2', 'r16i1p1f2', 'r2i1p1f2', 'r27i1p1f2', 'r24i1p1f2', 'r26i1p1f2', 'r19i1p1f2', 'r21i1p1f2', 'r18i1p1f2', 'r14i1p1f2', 'r6i1p1f2', 'r10i1p1f2', 'r28i1p1f2', 'r29i1p1f2', 'r20i1p1f2'],
    'HadGEM3-GC31-LL': ['r2i1p1f3', 'r3i1p1f3', 'r1i1p1f3', 'r4i1p1f3'],
    'FGOALS-f3-L': ['r1i1p1f1'],
    'KIOST-ESM': ['r1i1p1f1'],
    'NorESM2-LM': ['r3i1p1f1', 'r1i1p1f1', 'r2i1p1f1'],
    'ACCESS-CM2': ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1'],
    'NorESM2-MM': ['r1i1p1f1', 'r2i1p1f1'],
    'CNRM-CM6-1-HR': ['r1i1p1f2'],
    'KACE-1-0-G': ['r2i1p1f1', 'r1i1p1f1', 'r3i1p1f1'],
    'GISS-E2-1-G': ['r5i1p5f2', 'r6i1p5f2', 'r7i1p1f2', 'r8i1p5f2', 'r6i1p1f2', 'r8i1p1f2', 'r9i1p1f2', 'r9i1p5f2', 'r7i1p5f2', 'r5i1p3f1', 'r4i1p3f1', 'r3i1p5f2', 'r4i1p5f1', 'r5i1p5f1', 'r4i1p5f2', 'r10i1p1f2', 'r10i1p5f2', 'r3i1p3f1', 'r1i1p5f2', 'r1i1p5f1', 'r1i1p3f1', 'r3i1p5f1', 'r2i1p3f1', 'r2i1p5f1', 'r2i1p5f2'],
    'FIO-ESM-2-0': ['r3i1p1f1', 'r1i1p1f1', 'r2i1p1f1'],
    'EC-Earth3-Veg': ['r6i1p1f1', 'r1i1p1f1', 'r5i1p1f1', 'r4i1p1f1', 'r3i1p1f1', 'r14i1p1f1', 'r12i1p1f1'],
    'EC-Earth3': ['r11i1p1f1', 'r15i1p1f1', 'r6i1p1f1', 'r13i1p1f1', 'r9i1p1f1', 'r25i1p1f1', 'r1i1p1f1', 'r12i1p1f1', 'r22i1p1f1', 'r24i1p1f1', 'r23i1p1f1', 'r4i1p1f1', 'r27i1p1f2', 'r25i1p1f2', 'r29i1p1f2', 'r19i1p1f2', 'r12i1p1f2', 'r17i1p1f2', 'r15i1p1f2', 'r21i1p1f2', 'r23i1p1f2', 'r2i1p1f2', 'r8i1p1f2', 'r1i1p1f2', 'r30i1p1f2', 'r2i1p1f1', 'r7i1p1f1', 'r14i1p1f1', 'r10i1p1f1', 'r16i1p1f1', 'r17i1p1f1', 'r18i1p1f1', 'r19i1p1f1', 'r21i1p1f1', 'r124i1p1f1', 'r118i1p1f1', 'r121i1p1f1', 'r117i1p1f1', 'r123i1p1f1', 'r120i1p1f1', 'r122i1p1f1', 'r119i1p1f1', 'r103i1p1f1', 'r104i1p1f1', 'r105i1p1f1', 'r111i1p1f1', 'r106i1p1f1', 'r114i1p1f1', 'r116i1p1f1', 'r115i1p1f1', 'r102i1p1f1', 'r113i1p1f1', 'r107i1p1f1', 'r112i1p1f1', 'r109i1p1f1', 'r101i1p1f1', 'r108i1p1f1', 'r110i1p1f1'],
    'CIESM': ['r1i1p1f1'],
    'CESM2': ['r10i1p1f1', 'r4i1p1f1', 'r11i1p1f1'],
    'CMCC-CM2-SR5': ['r1i1p1f1'],
    'IITM-ESM': ['r1i1p1f1'],
    'E3SM-1-1': ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1', 'r4i1p1f1', 'r6i1p1f1', 'r8i1p1f1', 'r7i1p1f1', 'r9i1p1f1', 'r10i1p1f1', 'r5i1p1f1'],
    'EC-Earth3-Veg-LR': ['r2i1p1f1', 'r1i1p1f1', 'r3i1p1f1'],
    'TaiESM1': ['r1i1p1f1'],
    'EC-Earth3-CC': ['r1i1p1f1'],
    'CMCC-ESM2': ['r1i1p1f1']
}

incomplete_zg700_ssp245 = {
    'CNRM-CM6-1': ['r7i1p1f2', 'r8i1p1f2', 'r10i1p1f2', 'r9i1p1f2'],
    'UKESM1-0-LL': ['r11i1p1f2', 'r17i1p1f2', 'r16i1p1f2', 'r19i1p1f2', 'r12i1p1f2', 'r10i1p1f2', 'r18i1p1f2', 'r9i1p1f2'],
    'HadGEM3-GC31-LL': ['r3i1p1f3', 'r4i1p1f3', 'r2i1p1f3'],
    'E3SM-1-1': ['r1i1p1f1']
}

for k, v in modmembers_zg700_ssp245.items():
    if k in incomplete_zg700_ssp245.keys():
        modmembers_zg700_ssp245[k] = [x for x in modmembers_zg700_ssp245[k] if x not in incomplete_zg700_ssp245[k]]

In [15]:
for key in modmembers_tas_ssp245:
    if key not in modmembers_pr_ssp245.keys(): print(key)
    print(key, set(modmembers_tas_ssp245[key]) ^ set(modmembers_pr_ssp245[key]))

GFDL-CM4 set()
GFDL-ESM4 set()
IPSL-CM6A-LR set()
CNRM-CM6-1 set()
MRI-ESM2-0 set()
BCC-CSM2-MR set()
CNRM-ESM2-1 set()
CanESM5 set()
CanESM5-CanOE set()
UKESM1-0-LL set()
AWI-CM-1-1-MR set()
INM-CM4-8 set()
INM-CM5-0 set()
MIROC6 set()
CAMS-CSM1-0 set()
MPI-ESM1-2-LR set()
MPI-ESM1-2-HR set()
NESM3 set()
CESM2-WACCM set()
FGOALS-g3 set()
MIROC-ES2L set()
HadGEM3-GC31-LL set()
FGOALS-f3-L set()
NorESM2-LM set()
ACCESS-CM2 set()
NorESM2-MM set()
CNRM-CM6-1-HR set()
KACE-1-0-G set()
FIO-ESM-2-0 set()
GISS-E2-1-G set()
GISS-E2-1-H set()
EC-Earth3-Veg set()
EC-Earth3 {'r6i1p1f2'}
CIESM set()
CESM2 set()
CMCC-CM2-SR5 set()
IITM-ESM set()
E3SM-1-1 set()
EC-Earth3-Veg-LR set()
TaiESM1 set()
CAS-ESM2-0 set()
EC-Earth3-CC set()
CMCC-ESM2 set()
KIOST-ESM set()
ACCESS-ESM1-5 set()


In [5]:
models_rldscs = os.listdir('/d5/tessj/data/CMIP6/cmip6-esgf-globus/historical-processed/')
modmembers_rldscs = {}
modgr_rldscs = {}


for model in models_rldscs[:]:
    members_hist = [file.rsplit('_')[-2] for file in os.listdir(f'/d5/tessj/data/CMIP6/cmip6-esgf-globus/historical-processed/{model}')]
    members_ssp245 = [file.rsplit('_')[-2] for file in os.listdir(f'/d5/tessj/data/CMIP6/cmip6-esgf-globus/ssp245-processed/{model}')]
    members = list(set(members_hist) & set(members_ssp245))
    modmembers_rldscs[model] = members

#modmembers_tas_ssp585


In [16]:
# in case modmembers dict isn't in line with actual directory of members (bc some member_ids had duplicates in the original count due to being on both gr and gn grids)
# this is the real directory of members that we have (which makes MRI-ESM2-0 have actually only 6 ens members)
modmembers_nodupes_tas_ssp245 = {}
modmembers_tas_historical = {}
modmembers_pr_historical = {}


for model in list(modmembers_tas_ssp245.keys())[:]:
    model_directories = os.listdir(f'/d1/tessj/data/CMIP6/for_tess/{model}/') + os.listdir(f'/d3/tessj/data/CMIP6/for_tess/{model}/') + os.listdir(f'/d4/tessj/data/CMIP6/for_tess/{model}/')
    members = [filename[len(model + 'tas_Amon_ssp245__'):-len('_20150101-20241231_regrid.nc')] for filename in model_directories if (filename[-7:] == 'grid.nc') & (filename[:3]=='tas') & ('ssp245' in filename)]
    modmembers_nodupes_tas_ssp245[model] = list(set([member for member in members if member[0] == 'r']))
#modmembers_tas_ssp585

for model in list(modmembers_tas_ssp245.keys())[:]:
    model_directories = os.listdir(f'/d1/tessj/data/CMIP6/for_tess/{model}/') + os.listdir(f'/d3/tessj/data/CMIP6/for_tess/{model}/') + os.listdir(f'/d4/tessj/data/CMIP6/for_tess/{model}/')

    members = [filename[len(model + 'tas_Amon_historical__'):-len('_18500101-20241231_regrid.nc')] for filename in model_directories if (filename[-7:] == 'grid.nc') & (filename[:3]=='tas') & ('historical' in filename)]
    modmembers_tas_historical[model] = list(set([member for member in members if member[0] == 'r']))

for model in list(modmembers_pr_ssp245.keys())[:]:
    model_directories = os.listdir(f'/d1/tessj/data/CMIP6/for_tess/{model}/') + os.listdir(f'/d3/tessj/data/CMIP6/for_tess/{model}/') + os.listdir(f'/d4/tessj/data/CMIP6/for_tess/{model}/')

    members = [filename[len(model + 'pr_Amon_historical__'):-len('_18500101-20241231_regrid.nc')] for filename in model_directories if (filename[-7:] == 'grid.nc') & (filename[:2]=='pr') & ('historical' in filename)]
    modmembers_pr_historical[model] = list(set([member for member in members if member[0] == 'r']))



In [17]:
# EC Earth3 has some members that don't go all the way back to 1850 
incomplete_tas_historical = {'EC-Earth3': ['r106i1p1f1',
  'r114i1p1f1',
  'r110i1p1f1',
  'r112i1p1f1',
  'r128i1p1f1',
  'r119i1p1f1',
  'r150i1p1f1',
  'r137i1p1f1',
  'r123i1p1f1',
  'r129i1p1f1',
  'r138i1p1f1',
  'r120i1p1f1',
  'r104i1p1f1',
  'r144i1p1f1',
  'r131i1p1f1',
  'r115i1p1f1',
  'r122i1p1f1',
  'r136i1p1f1',
  'r133i1p1f1','r116i1p1f1',
  'r149i1p1f1',
  'r103i1p1f1',
  'r130i1p1f1',
  'r125i1p1f1',
  'r132i1p1f1',
  'r101i1p1f1',
  'r135i1p1f1',
  'r140i1p1f1',
  'r147i1p1f1',
  'r143i1p1f1',
  'r121i1p1f1',
  'r109i1p1f1',
  'r145i1p1f1',
  'r148i1p1f1',
  'r102i1p1f1',
  'r139i1p1f1',
  'r134i1p1f1',
  'r117i1p1f1',
  'r118i1p1f1',
  'r108i1p1f1',
  'r113i1p1f1',
  'r127i1p1f1',
  'r107i1p1f1',
  'r142i1p1f1',
  'r111i1p1f1',
  'r126i1p1f1',
  'r141i1p1f1',
  'r105i1p1f1',
  'r146i1p1f1',
  'r124i1p1f1']}

for k, v in modmembers_tas_historical.items():
    if k in incomplete_tas_historical.keys():
        modmembers_tas_historical[k] = [x for x in modmembers_tas_historical[k] if x not in incomplete_tas_historical[k]]

In [18]:
incomplete_pr_historical = {'NorESM2-LM': ['r1i1p1f1'],
 'EC-Earth3': ['r106i1p1f1',
  'r116i1p1f1',
  'r110i1p1f1',
  'r112i1p1f1',
  'r128i1p1f1',
  'r114i1p1f1',
  'r123i1p1f1',
  'r137i1p1f1',
  'r119i1p1f1',
  'r129i1p1f1',
  'r138i1p1f1',
  'r120i1p1f1',
  'r104i1p1f1',
  'r144i1p1f1',
  'r131i1p1f1',
  'r115i1p1f1',
  'r150i1p1f1',
  'r122i1p1f1',
  'r136i1p1f1', 'r146i1p1f1',
  'r133i1p1f1',
  'r149i1p1f1',
  'r103i1p1f1',
  'r130i1p1f1',
  'r125i1p1f1',
  'r132i1p1f1',
  'r101i1p1f1',
  'r135i1p1f1',
  'r140i1p1f1',
  'r121i1p1f1',
  'r143i1p1f1',
  'r147i1p1f1',
  'r109i1p1f1',
  'r145i1p1f1',
  'r148i1p1f1',
  'r102i1p1f1',
  'r139i1p1f1',
  'r117i1p1f1',
  'r134i1p1f1',
  'r118i1p1f1',
  'r108i1p1f1',
  'r113i1p1f1',
  'r127i1p1f1',
  'r107i1p1f1',
  'r142i1p1f1',
  'r111i1p1f1',
  'r126i1p1f1',
  'r141i1p1f1',
  'r105i1p1f1',
  'r124i1p1f1']}
for k, v in modmembers_pr_historical.items():
    if k in incomplete_pr_historical.keys():
        modmembers_pr_historical[k] = [x for x in modmembers_pr_historical[k] if x not in incomplete_pr_historical[k]]

In [21]:
# mod members in both hist and ssp245
modmembers_tas= {}
for model in list(modmembers_tas_ssp245.keys()):
    if len((set(modmembers_tas_ssp245[model]) & set(modmembers_tas_historical[model])))>0:
        modmembers_tas[model] = list(set(modmembers_tas_ssp245[model]) & set(modmembers_tas_historical[model]))
        
modmembers_pr= {}
for model in list(modmembers_pr_ssp245.keys()):
    if len((set(modmembers_pr_ssp245[model]) & set(modmembers_pr_historical[model])))>0:
        modmembers_pr[model] = list(set(modmembers_pr_ssp245[model]) & set(modmembers_pr_historical[model]))

In [22]:
modmembers_zg700= {}
for model in list(modmembers_zg700_ssp245.keys()):
    if len((set(modmembers_zg700_ssp245[model]) & set(modmembers_zg700_hist[model])))>0:
        modmembers_zg700[model] = list(set(modmembers_zg700_ssp245[model]) & set(modmembers_zg700_hist[model]))

In [23]:
incomplete_runs_huss_hist = {'EC-Earth3': ['r111i1p1f1', 'r149i1p1f1', 'r147i1p1f1', 'r115i1p1f1', 'r150i1p1f1', 'r145i1p1f1', 'r120i1p1f1', 'r123i1p1f1', 'r104i1p1f1', 'r107i1p1f1', 'r126i1p1f1', 'r140i1p1f1', 'r113i1p1f1', 'r102i1p1f1', 'r143i1p1f1', 'r101i1p1f1', 'r128i1p1f1', 'r110i1p1f1', 'r127i1p1f1', 'r138i1p1f1', 'r136i1p1f1', 'r109i1p1f1', 'r103i1p1f1', 'r121i1p1f1', 'r146i1p1f1', 'r122i1p1f1', 'r132i1p1f1', 'r106i1p1f1', 'r137i1p1f1', 'r117i1p1f1', 'r139i1p1f1', 'r135i1p1f1', 'r144i1p1f1', 'r129i1p1f1', 'r142i1p1f1', 'r125i1p1f1', 'r105i1p1f1', 'r148i1p1f1', 'r116i1p1f1', 'r119i1p1f1', 'r118i1p1f1', 'r141i1p1f1', 'r131i1p1f1', 'r134i1p1f1', 'r124i1p1f1', 'r133i1p1f1', 'r112i1p1f1', 'r130i1p1f1', 'r108i1p1f1', 'r114i1p1f1']}

incomplete_runs_huss_ssp245 = {'CNRM-CM6-1': ['r9i1p1f2', 'r7i1p1f2', 'r10i1p1f2', 'r8i1p1f2'],
 'UKESM1-0-LL': ['r12i1p1f2'],
 'HadGEM3-GC31-LL': ['r3i1p1f3', 'r2i1p1f3', 'r4i1p1f3'],
 'E3SM-1-1': ['r1i1p1f1']}

for k, v in modmembers_huss_ssp245.items():
    if k in incomplete_runs_huss_ssp245.keys():
        modmembers_huss_ssp245[k] = [x for x in modmembers_huss_ssp245[k] if x not in incomplete_runs_huss_ssp245[k]]
        
for k, v in modmembers_huss_hist.items():
    if k in incomplete_runs_huss_hist.keys():
        modmembers_huss_hist[k] = [x for x in modmembers_huss_hist[k] if x not in incomplete_runs_huss_hist[k]]

In [25]:
incomplete_runs_hus300_hist = {'EC-Earth3': ['r101i1p1f1', 'r112i1p1f1', 'r119i1p1f1', 'r102i1p1f1', 'r124i1p1f1', 'r106i1p1f1', 'r117i1p1f1', 'r111i1p1f1', 'r114i1p1f1', 'r108i1p1f1', 'r125i1p1f1', 'r122i1p1f1', 'r107i1p1f1', 'r121i1p1f1', 'r105i1p1f1', 'r110i1p1f1', 'r115i1p1f1', 'r123i1p1f1', 'r104i1p1f1', 'r120i1p1f1', 'r118i1p1f1', 'r113i1p1f1', 'r103i1p1f1', 'r109i1p1f1', 'r116i1p1f1']}
incomplete_runs_hus300_ssp245 = {'CNRM-CM6-1': ['r9i1p1f2', 'r7i1p1f2', 'r10i1p1f2', 'r8i1p1f2'], 
                               'UKESM1-0-LL': ['r12i1p1f2'], 
                               'HadGEM3-GC31-LL': ['r3i1p1f3', 'r2i1p1f3', 'r4i1p1f3'], 
                               'E3SM-1-1': ['r1i1p1f1']}


for k, v in modmembers_hus300_ssp245.items():
    if k in incomplete_runs_hus300_ssp245.keys():
        modmembers_hus300_ssp245[k] = [x for x in modmembers_hus300_ssp245[k] if x not in incomplete_runs_hus300_ssp245[k]]
        
for k, v in modmembers_hus300_hist.items():
    if k in incomplete_runs_hus300_hist.keys():
        modmembers_hus300_hist[k] = [x for x in modmembers_hus300_hist[k] if x not in incomplete_runs_hus300_hist[k]]

In [28]:
modmembers_huss= {}
for model in list(modmembers_huss_ssp245.keys()):
    if len((set(modmembers_huss_ssp245[model]) & set(modmembers_huss_hist[model])))>0:
        modmembers_huss[model] = list(set(modmembers_huss_ssp245[model]) & set(modmembers_huss_hist[model]))

In [29]:
modmembers_hus300= {}
for model in list(modmembers_hus300_ssp245.keys()):
    if len((set(modmembers_hus300_ssp245[model]) & set(modmembers_hus300_hist[model])))>0:
        modmembers_hus300[model] = list(set(modmembers_hus300_ssp245[model]) & set(modmembers_hus300_hist[model]))

In [30]:
modmembers_hus500= modmembers_hus300
modmembers_hus700= modmembers_hus300


In [31]:
incomplete_runs_prw_hist = {'EC-Earth3': ['r11i1p1f1', 'r13i1p1f1', 'r111i1p1f1', 'r149i1p1f1', 'r147i1p1f1', 'r115i1p1f1', 'r150i1p1f1', 'r145i1p1f1', 'r120i1p1f1', 'r123i1p1f1', 'r104i1p1f1', 'r107i1p1f1', 'r126i1p1f1', 'r140i1p1f1', 'r113i1p1f1', 'r102i1p1f1', 'r143i1p1f1', 'r101i1p1f1', 'r128i1p1f1', 'r110i1p1f1', 'r127i1p1f1', 'r138i1p1f1', 'r136i1p1f1', 'r109i1p1f1', 'r103i1p1f1', 'r121i1p1f1', 'r146i1p1f1', 'r122i1p1f1', 'r132i1p1f1', 'r106i1p1f1', 'r137i1p1f1', 'r117i1p1f1', 'r139i1p1f1', 'r135i1p1f1', 'r144i1p1f1', 'r129i1p1f1', 'r142i1p1f1', 'r125i1p1f1', 'r105i1p1f1', 'r148i1p1f1', 'r116i1p1f1', 'r119i1p1f1', 'r118i1p1f1', 'r141i1p1f1', 'r131i1p1f1', 'r134i1p1f1', 'r124i1p1f1', 'r133i1p1f1', 'r112i1p1f1', 'r130i1p1f1', 'r108i1p1f1', 'r114i1p1f1'],
                           'FGOALS-g3':['r1i1p1f1', 'r3i1p1f1', 'r2i1p1f1', 'r4i1p1f1', 'r6i1p1f1', 'r5i1p1f1']}

incomplete_runs_prw_ssp245 = {'CNRM-CM6-1': ['r9i1p1f2', 'r7i1p1f2', 'r10i1p1f2', 'r8i1p1f2'],
 'UKESM1-0-LL': ['r12i1p1f2'],
 'HadGEM3-GC31-LL': ['r3i1p1f3', 'r2i1p1f3', 'r4i1p1f3'],
 'E3SM-1-1': ['r1i1p1f1']}

for k, v in modmembers_prw_ssp245.items():
    if k in incomplete_runs_prw_ssp245.keys():
        modmembers_prw_ssp245[k] = [x for x in modmembers_prw_ssp245[k] if x not in incomplete_runs_prw_ssp245[k]]
        
for k, v in modmembers_prw_hist.items():
    if k in incomplete_runs_prw_hist.keys():
        modmembers_prw_hist[k] = [x for x in modmembers_prw_hist[k] if x not in incomplete_runs_prw_hist[k]]

In [32]:
modmembers_prw= {}
for model in list(modmembers_prw_ssp245.keys()):
    if len((set(modmembers_prw_ssp245[model]) & set(modmembers_prw_hist[model])))>0:
        modmembers_prw[model] = list(set(modmembers_prw_ssp245[model]) & set(modmembers_prw_hist[model]))

In [34]:
incomplete_runs_hfls_hist = {}
incomplete_runs_hfls_ssp245 = {'CNRM-CM6-1': ['r7i1p1f2', 'r9i1p1f2', 'r8i1p1f2', 'r10i1p1f2'],
 'HadGEM3-GC31-LL': ['r3i1p1f3', 'r2i1p1f3', 'r4i1p1f3']}

for k, v in modmembers_hfls_ssp245.items():
    if k in incomplete_runs_hfls_ssp245.keys():
        modmembers_hfls_ssp245[k] = [x for x in modmembers_hfls_ssp245[k] if x not in incomplete_runs_hfls_ssp245[k]]
        
for k, v in modmembers_hfls_hist.items():
    if k in incomplete_runs_hfls_hist.keys():
        modmembers_hfls_hist[k] = [x for x in modmembers_hfls_hist[k] if x not in incomplete_runs_hfls_hist[k]]

In [35]:
modmembers_hfls= {}
for model in list(modmembers_hfls_ssp245.keys()):
    if len((set(modmembers_hfls_ssp245[model]) & set(modmembers_hfls_hist[model])))>0:
        modmembers_hfls[model] = list(set(modmembers_hfls_ssp245[model]) & set(modmembers_hfls_hist[model]))

In [36]:
incomplete_runs_hfss_hist = {}
incomplete_runs_hfss_ssp245 = {'HadGEM3-GC31-LL': ['r3i1p1f3', 'r2i1p1f3', 'r4i1p1f3'],
 'CNRM-CM6-1': ['r7i1p1f2', 'r9i1p1f2', 'r8i1p1f2', 'r10i1p1f2']}

for k, v in modmembers_hfss_ssp245.items():
    if k in incomplete_runs_hfss_ssp245.keys():
        modmembers_hfss_ssp245[k] = [x for x in modmembers_hfss_ssp245[k] if x not in incomplete_runs_hfss_ssp245[k]]
        
for k, v in modmembers_hfss_hist.items():
    if k in incomplete_runs_hfss_hist.keys():
        modmembers_hfss_hist[k] = [x for x in modmembers_hfss_hist[k] if x not in incomplete_runs_hfss_hist[k]]

In [37]:
modmembers_hfss= {}
for model in list(set(modmembers_hfss_hist.keys()) & set(modmembers_hfss_ssp245.keys())):
    if len((set(modmembers_hfss_ssp245[model]) & set(modmembers_hfss_hist[model])))>0:
        modmembers_hfss[model] = list(set(modmembers_hfss_ssp245[model]) & set(modmembers_hfss_hist[model]))

In [38]:
incomplete_runs_rsds_hist = {}
incomplete_runs_rsds_ssp245 = {'CNRM-CM6-1': ['r7i1p1f2', 'r9i1p1f2', 'r8i1p1f2', 'r10i1p1f2'],
 'UKESM1-0-LL': ['r16i1p1f2', 'r19i1p1f2',  'r9i1p1f2',  'r18i1p1f2',  'r17i1p1f2',  'r11i1p1f2',  'r10i1p1f2',  'r12i1p1f2'],
 'HadGEM3-GC31-LL': ['r3i1p1f3', 'r2i1p1f3', 'r4i1p1f3']}

for k, v in modmembers_rsds_ssp245.items():
    if k in incomplete_runs_rsds_ssp245.keys():
        modmembers_rsds_ssp245[k] = [x for x in modmembers_rsds_ssp245[k] if x not in incomplete_runs_rsds_ssp245[k]]
        
for k, v in modmembers_rsds_hist.items():
    if k in incomplete_runs_rsds_hist.keys():
        modmembers_rsds_hist[k] = [x for x in modmembers_rsds_hist[k] if x not in incomplete_runs_rsds_hist[k]]

In [39]:
modmembers_rsds= {}
for model in list(modmembers_rsds_ssp245.keys()):
    if len((set(modmembers_rsds_ssp245[model]) & set(modmembers_rsds_hist[model])))>0:
        modmembers_rsds[model] = list(set(modmembers_rsds_ssp245[model]) & set(modmembers_rsds_hist[model]))

In [40]:
incomplete_runs_rsus_hist = {}
incomplete_runs_rsus_ssp245 = {'CNRM-CM6-1': ['r7i1p1f2', 'r9i1p1f2', 'r8i1p1f2', 'r10i1p1f2'],
 'UKESM1-0-LL': ['r16i1p1f2', 'r19i1p1f2',  'r9i1p1f2',  'r18i1p1f2',  'r17i1p1f2',  'r11i1p1f2',  'r10i1p1f2',  'r12i1p1f2'],
 'HadGEM3-GC31-LL': ['r3i1p1f3', 'r2i1p1f3', 'r4i1p1f3']}

for k, v in modmembers_rsus_ssp245.items():
    if k in incomplete_runs_rsus_ssp245.keys():
        modmembers_rsus_ssp245[k] = [x for x in modmembers_rsus_ssp245[k] if x not in incomplete_runs_rsus_ssp245[k]]
        
for k, v in modmembers_rsus_hist.items():
    if k in incomplete_runs_rsus_hist.keys():
        modmembers_rsus_hist[k] = [x for x in modmembers_rsus_hist[k] if x not in incomplete_runs_rsus_hist[k]]

In [41]:
modmembers_rsus= {}
for model in list(set(modmembers_rsus_hist.keys()) & set(modmembers_rsus_ssp245.keys())):
    if len((set(modmembers_rsus_ssp245[model]) & set(modmembers_rsus_hist[model])))>0:
        modmembers_rsus[model] = list(set(modmembers_rsus_ssp245[model]) & set(modmembers_rsus_hist[model]))

In [42]:
incomplete_runs_rsdt_hist = {}
incomplete_runs_rsdt_ssp245 = {'CNRM-CM6-1': ['r7i1p1f2', 'r9i1p1f2', 'r8i1p1f2', 'r10i1p1f2'],
 'UKESM1-0-LL': ['r16i1p1f2', 'r19i1p1f2',  'r9i1p1f2',  'r18i1p1f2',  'r17i1p1f2',  'r11i1p1f2',  'r10i1p1f2',  'r12i1p1f2'],
 'HadGEM3-GC31-LL': ['r3i1p1f3', 'r2i1p1f3', 'r4i1p1f3']}

for k, v in modmembers_rsdt_ssp245.items():
    if k in incomplete_runs_rsdt_ssp245.keys():
        modmembers_rsdt_ssp245[k] = [x for x in modmembers_rsdt_ssp245[k] if x not in incomplete_runs_rsdt_ssp245[k]]
        
for k, v in modmembers_rsdt_hist.items():
    if k in incomplete_runs_rsdt_hist.keys():
        modmembers_rsdt_hist[k] = [x for x in modmembers_rsdt_hist[k] if x not in incomplete_runs_rsdt_hist[k]]

In [43]:
modmembers_rsdt= {}
for model in list(modmembers_rsdt_ssp245.keys()):
    if len((set(modmembers_rsdt_ssp245[model]) & set(modmembers_rsdt_hist[model])))>0:
        modmembers_rsdt[model] = list(set(modmembers_rsdt_ssp245[model]) & set(modmembers_rsdt_hist[model]))

In [44]:
incomplete_runs_rsut_hist = {}
incomplete_runs_rsut_ssp245 = {'CNRM-CM6-1': ['r7i1p1f2', 'r9i1p1f2', 'r8i1p1f2', 'r10i1p1f2'],
 'UKESM1-0-LL': ['r16i1p1f2', 'r19i1p1f2',  'r9i1p1f2',  'r18i1p1f2',  'r17i1p1f2',  'r11i1p1f2',  'r10i1p1f2',  'r12i1p1f2'],
 'HadGEM3-GC31-LL': ['r3i1p1f3', 'r2i1p1f3', 'r4i1p1f3']}


for k, v in modmembers_rsut_ssp245.items():
    if k in incomplete_runs_rsut_ssp245.keys():
        modmembers_rsut_ssp245[k] = [x for x in modmembers_rsut_ssp245[k] if x not in incomplete_runs_rsut_ssp245[k]]
        
for k, v in modmembers_rsut_hist.items():
    if k in incomplete_runs_rsut_hist.keys():
        modmembers_rsut_hist[k] = [x for x in modmembers_rsut_hist[k] if x not in incomplete_runs_rsut_hist[k]]

In [45]:
modmembers_rsut= {}
for model in list(modmembers_rsut_ssp245.keys()):
    if len((set(modmembers_rsut_ssp245[model]) & set(modmembers_rsut_hist[model])))>0:
        modmembers_rsut[model] = list(set(modmembers_rsut_ssp245[model]) & set(modmembers_rsut_hist[model]))

In [46]:
incomplete_runs_clt_hist = {'EC-Earth3': ['r111i1p1f1', 'r149i1p1f1', 'r147i1p1f1', 'r115i1p1f1', 'r150i1p1f1', 'r145i1p1f1', 'r120i1p1f1', 'r123i1p1f1', 'r104i1p1f1', 'r107i1p1f1', 'r126i1p1f1', 'r140i1p1f1', 'r113i1p1f1', 'r102i1p1f1', 'r143i1p1f1', 'r101i1p1f1', 'r128i1p1f1', 'r110i1p1f1', 'r127i1p1f1', 'r138i1p1f1', 'r136i1p1f1', 'r109i1p1f1', 'r103i1p1f1', 'r121i1p1f1', 'r146i1p1f1', 'r122i1p1f1', 'r132i1p1f1', 'r106i1p1f1', 'r137i1p1f1', 'r117i1p1f1', 'r139i1p1f1', 'r135i1p1f1', 'r144i1p1f1', 'r129i1p1f1', 'r142i1p1f1', 'r125i1p1f1', 'r105i1p1f1', 'r148i1p1f1', 'r116i1p1f1', 'r119i1p1f1', 'r118i1p1f1', 'r141i1p1f1', 'r131i1p1f1', 'r134i1p1f1', 'r124i1p1f1', 'r133i1p1f1', 'r112i1p1f1', 'r130i1p1f1', 'r108i1p1f1', 'r114i1p1f1']}
incomplete_runs_clt_ssp245 = {'CNRM-CM6-1': ['r7i1p1f2', 'r9i1p1f2', 'r8i1p1f2', 'r10i1p1f2'],
 'UKESM1-0-LL': ['r11i1p1f2', 'r12i1p1f2'],
 'HadGEM3-GC31-LL': ['r3i1p1f3', 'r2i1p1f3', 'r4i1p1f3'],
 'E3SM-1-1': ['r1i1p1f1']}

for k, v in modmembers_clt_ssp245.items():
    if k in incomplete_runs_clt_ssp245.keys():
        modmembers_clt_ssp245[k] = [x for x in modmembers_clt_ssp245[k] if x not in incomplete_runs_clt_ssp245[k]]
        
for k, v in modmembers_clt_hist.items():
    if k in incomplete_runs_clt_hist.keys():
        modmembers_clt_hist[k] = [x for x in modmembers_clt_hist[k] if x not in incomplete_runs_clt_hist[k]]

In [47]:
modmembers_clt= {}
for model in list(modmembers_clt_ssp245.keys()):
    if len((set(modmembers_clt_ssp245[model]) & set(modmembers_clt_hist[model])))>0:
        modmembers_clt[model] = list(set(modmembers_clt_ssp245[model]) & set(modmembers_clt_hist[model]))

In [48]:
modmembers_lw= {}
for model in list(modmembers_rlds_hist.keys()):
    if ((model in modmembers_rlds_hist.keys()) & (model in modmembers_rlds_ssp245.keys())
        &(model in modmembers_rlus_hist.keys()) & (model in modmembers_rlus_ssp245.keys())
        &(model in modmembers_rlut_hist.keys()) & (model in modmembers_rlut_ssp245.keys())):
        if len(set(modmembers_rlds_hist[model]) & set(modmembers_rlds_ssp245[model]) & set(modmembers_rlus_ssp245[model]) & set(modmembers_rlus_ssp245[model])
               & set(modmembers_rlut_ssp245[model]) & set(modmembers_rlut_ssp245[model]))>0:
            modmembers_lw[model] = list(set(modmembers_rlds_hist[model]) & set(modmembers_rlds_ssp245[model]) 
                & set(modmembers_rlus_ssp245[model]) & set(modmembers_rlus_ssp245[model])
               & set(modmembers_rlut_ssp245[model]) & set(modmembers_rlut_ssp245[model]))
            
incomplete_runs_lw_ssp245 = {'CNRM-CM6-1': ['r9i1p1f2', 'r7i1p1f2', 'r10i1p1f2', 'r8i1p1f2'],
 'UKESM1-0-LL': ['r16i1p1f2',
  'r9i1p1f2',
  'r11i1p1f2',
  'r18i1p1f2',
  'r10i1p1f2',
  'r12i1p1f2',
  'r19i1p1f2',
  'r17i1p1f2']}

for k, v in modmembers_lw.items():
    if k in incomplete_runs_lw_ssp245.keys():
        modmembers_lw[k] = [x for x in modmembers_lw[k] if x not in incomplete_runs_lw_ssp245[k]]

In [49]:
modmembers_lwd= {}
for model in list(modmembers_rlds_hist.keys()):
    if ((model in modmembers_rlds_hist.keys()) & (model in modmembers_rlds_ssp245.keys())):
        if len(set(modmembers_rlds_hist[model]) & set(modmembers_rlds_ssp245[model]))>0:
            modmembers_lwd[model] = list(set(modmembers_rlds_hist[model]) & set(modmembers_rlds_ssp245[model]))
            
            
incomplete_runs_lwd_ssp245  =    {'CNRM-CM6-1': ['r9i1p1f2', 'r7i1p1f2', 'r10i1p1f2', 'r8i1p1f2'],
 'UKESM1-0-LL': ['r16i1p1f2',
  'r9i1p1f2',
  'r11i1p1f2',
  'r18i1p1f2',
  'r10i1p1f2',
  'r12i1p1f2',
  'r19i1p1f2',
  'r17i1p1f2'],
 'HadGEM3-GC31-LL': ['r3i1p1f3', 'r2i1p1f3', 'r4i1p1f3']}

for k, v in modmembers_lwd.items():
    if k in incomplete_runs_lwd_ssp245.keys():
        modmembers_lwd[k] = [x for x in modmembers_lwd[k] if x not in incomplete_runs_lwd_ssp245[k]]

In [50]:
for model in modmembers_lw:
    if len(modmembers_lw[model])>=3:
        print(model, len(modmembers_lw[model]))

IPSL-CM6A-LR 11
GISS-E2-1-G 14
CNRM-CM6-1 6
CNRM-ESM2-1 9
MRI-ESM2-0 5
CESM2-WACCM 3
CESM2 3
MIROC6 44
UKESM1-0-LL 6
CanESM5 50
CanESM5-CanOE 3
HadGEM3-GC31-LL 4
EC-Earth3 6
MPI-ESM1-2-LR 10
NorESM2-LM 3
FGOALS-g3 4
KACE-1-0-G 3
FIO-ESM-2-0 3


In [51]:
modmembers_geq3_tas = {}
modmembers_geq3_pr = {}
modmembers_geq3_zg700 = {}
modmembers_geq3_lw = {}
modmembers_geq3_lwd = {}
modmembers_geq3_huss = {}
modmembers_geq3_hus300 = {}


modmembers_geq3_hfls = {}
modmembers_geq3_hfss = {}
modmembers_geq3_rsds = {}
modmembers_geq3_rsus = {}
modmembers_geq3_rsut = {}
modmembers_geq3_rsdt = {}
modmembers_geq3_clt = {}
modmembers_geq3_prw = {}


for model in modmembers_tas.keys():
    if len(modmembers_tas[model])>=3:
        modmembers_geq3_tas[model] = modmembers_tas[model]
        modmembers_geq3_pr[model] = modmembers_pr[model]
        if model in modmembers_zg700.keys():
            modmembers_geq3_zg700[model] = modmembers_zg700[model]
        if model in modmembers_lw.keys():
            modmembers_geq3_lw[model] = modmembers_lw[model]
        if model in modmembers_lwd.keys():
            modmembers_geq3_lwd[model] = modmembers_lwd[model]
        if model in modmembers_huss.keys():
            modmembers_geq3_huss[model] = modmembers_huss[model]
        if model in modmembers_hus300.keys():
            modmembers_geq3_hus300[model] = modmembers_hus300[model]
        if model in modmembers_hfls.keys():
            modmembers_geq3_hfls[model] = modmembers_hfls[model]
        if model in modmembers_hfss.keys():
            modmembers_geq3_hfss[model] = modmembers_hfss[model]            
        if model in modmembers_rsds.keys():
            modmembers_geq3_rsds[model] = modmembers_rsds[model]
        if model in modmembers_rsus.keys():
            modmembers_geq3_rsus[model] = modmembers_rsus[model]  
        if model in modmembers_rsdt.keys():
            modmembers_geq3_rsdt[model] = modmembers_rsdt[model]
        if model in modmembers_rsut.keys():
            modmembers_geq3_rsut[model] = modmembers_rsut[model]  
        if model in modmembers_clt.keys():
            modmembers_geq3_clt[model] = modmembers_clt[model]  
        if model in modmembers_prw.keys():
            modmembers_geq3_prw[model] = modmembers_prw[model]  
                
            print(model, len(modmembers_tas[model]))#, len(modmembers_pr[model]), len(modmembers_zg700[model]), len(modmembers_lw[model]), len(modmembers_lwd[model]),len(modmembers_huss[model]))

modmembers_geq3_hus500 = modmembers_geq3_hus300
modmembers_geq3_hus700 = modmembers_geq3_hus300

GFDL-ESM4 3
IPSL-CM6A-LR 11
CNRM-CM6-1 6
MRI-ESM2-0 5
CNRM-ESM2-1 9
CanESM5 50
CanESM5-CanOE 3
UKESM1-0-LL 14
MIROC6 50
MPI-ESM1-2-LR 10
CESM2-WACCM 3
MIROC-ES2L 30
NorESM2-LM 3
ACCESS-CM2 5
KACE-1-0-G 3
GISS-E2-1-G 14
EC-Earth3-Veg 8
EC-Earth3 22
CESM2 3
EC-Earth3-Veg-LR 3


In [11]:
# using any 1-degree regridded CMIP6 model to make land mask
tas_test = xr.open_dataset(f'/d3/tessj/data/CMIP6/for_tess/ACCESS-ESM1-5/tas_Amon_ACCESS-ESM1-5_historical_r3i1p1f1_18500101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-31'))

# using the ERA5 land-sea mask
landmask = fix_coords_lon(xr.open_dataarray('/home/tessj/wildfire-clim/land_sea_mask.nc').isel(time=-1).load()).rename({'latitude':'lat', 'longitude':'lon'}).drop(['expver','time'])
landmask = landmask.interp_like(tas_test)
landmask = landmask.where(landmask>0.4, np.nan)
landmask = landmask/landmask

In [12]:
wusbox = [-125, -102, 32,  49]

weights = np.cos(np.deg2rad(landmask.lat))

## Open model data into lists

In [184]:
modmembers_tas_hist_disk = {}
modmembers_tas_ssp245_disk = {} 

for model in modmembers_tas:
    memberdict = {}
    for member_id in modmembers_tas[model]:
        if f'tas_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc' in os.listdir(f'/d3/tessj/data/CMIP6/for_tess/{model}/'):
            memberdict[member_id] = 'd3'
        elif f'tas_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc' in os.listdir(f'/d1/tessj/data/CMIP6/for_tess/{model}/'):
            memberdict[member_id] = 'd1'        
        elif f'tas_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc' in os.listdir(f'/d4/tessj/data/CMIP6/for_tess/{model}/'):
            memberdict[member_id] = 'd4'
    modmembers_tas_hist_disk[model] =memberdict

for model in modmembers_tas:
    memberdict = {}
    for member_id in modmembers_tas[model]:
        if f'tas_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc' in os.listdir(f'/d3/tessj/data/CMIP6/for_tess/{model}/'):
            memberdict[member_id] = 'd3'
        elif f'tas_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc' in os.listdir(f'/d1/tessj/data/CMIP6/for_tess/{model}/'):
            memberdict[member_id] = 'd1'        
        elif f'tas_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc' in os.listdir(f'/d4/tessj/data/CMIP6/for_tess/{model}/'):
            memberdict[member_id] = 'd4'
    modmembers_tas_ssp245_disk[model] =memberdict

In [319]:
modmembers_pr_hist_disk = {}
modmembers_pr_ssp245_disk = {} 

for model in modmembers_pr:
    memberdict = {}
    for member_id in modmembers_pr[model]:
        if f'pr_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc' in os.listdir(f'/d3/tessj/data/CMIP6/for_tess/{model}/'):
            memberdict[member_id] = 'd3'
        elif f'pr_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc' in os.listdir(f'/d1/tessj/data/CMIP6/for_tess/{model}/'):
            memberdict[member_id] = 'd1'        
        elif f'pr_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc' in os.listdir(f'/d4/tessj/data/CMIP6/for_tess/{model}/'):
            memberdict[member_id] = 'd4'
    modmembers_pr_hist_disk[model] =memberdict

for model in modmembers_pr:
    memberdict = {}
    for member_id in modmembers_pr[model]:
        if f'pr_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc' in os.listdir(f'/d3/tessj/data/CMIP6/for_tess/{model}/'):
            memberdict[member_id] = 'd3'
        elif f'pr_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc' in os.listdir(f'/d1/tessj/data/CMIP6/for_tess/{model}/'):
            memberdict[member_id] = 'd1'        
        elif f'pr_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc' in os.listdir(f'/d4/tessj/data/CMIP6/for_tess/{model}/'):
            memberdict[member_id] = 'd4'
    modmembers_pr_ssp245_disk[model] =memberdict

In [ ]:
wusbox = [-125, -102, 32,  49]

weights = np.cos(np.deg2rad(landmask.lat))

In [ ]:
models_tas_avg_list = []
#models_tas_wus_list = []
models_tas_na_list = []


for model in list(modmembers_geq3_tas.keys()):
    print(model, len(modmembers_tas[model]))
    model_tas_avg_list = []
    #model_tas_wus_list = []
    model_tas_na_list = []
    

    for member_id in modmembers_tas[model][:]:
        disk_hist = modmembers_tas_hist_disk[model][member_id]
        disk_ssp245 = modmembers_tas_ssp245_disk[model][member_id]
        tas_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/tas_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        tas_ss245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/tas_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')
        if 'height' in list(tas_hist.coords):
            tas_total = xr.concat([tas_hist, tas_ss245], dim='time').drop('height')
        else:             
            tas_total = xr.concat([tas_hist, tas_ss245], dim='time')
        tas_globland_mean = tas_total.where(landmask==landmask).weighted(weights).mean(['lon','lat']).rename({'tas':'tas_globland'})
        tas_na = tas_total.sel(lat=slice(0,65), lon=slice(-180,-80))
        tas_wus = tas_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
        tas_wus_mean = tas_wus.weighted(weights).mean(['lon','lat']).rename({'tas':'tas_wus'})
        tas_avgs = xr.merge([tas_globland_mean, tas_wus_mean])#.assign_coords({'membermod':f'{model}_{member_id}'})
        model_tas_avg_list.append(tas_avgs)
        #model_tas_wus_list.append(tas_wus)
        model_tas_na_list.append(tas_na)

    models_tas_avg_list.append(sum(model_tas_avg_list)/len(model_tas_avg_list))
    #models_tas_wus_list.append(sum(model_tas_wus_list)/len(model_tas_wus_list))
    models_tas_na_list.append(sum(model_tas_na_list)/len(model_tas_na_list))


GFDL-ESM4 3
IPSL-CM6A-LR 11
CNRM-CM6-1 6
MRI-ESM2-0 5
CNRM-ESM2-1 9
CanESM5 50
CanESM5-CanOE 3
UKESM1-0-LL 14
MIROC6 50
MPI-ESM1-2-LR 10
CESM2-WACCM 3
FGOALS-g3 4


In [87]:
#models_lw_avg_list = []
#models_lw_na_list = []


for model in list(modmembers_geq3_lw.keys()):
#for (i, model) in [(2, 'CNRM-CM6-1'), (7,'UKESM1-0-LL')]:

    print(model, len(modmembers_lw[model]))
    model_lw_avg_list = []
    model_lw_na_list = []
    

    for member_id in modmembers_lw[model][:]:
        disk_hist = 'd4'
        disk_ssp245 = 'd4'
        rlds_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/rlds_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        rlds_ssp245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/rlds_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')
        
        rlus_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/rlus_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        rlus_ssp245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/rlus_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')
        
        rlut_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/rlut_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        rlut_ssp245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/rlut_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')
        
        # net longwave absorbed by atmosphere is net longwave up at surface minus net longwave out at toa 
        nlw_hist = (rlus_hist.rlus - rlds_hist.rlds) - rlut_hist.rlut
        nlw_ssp245 = (rlus_ssp245.rlus - rlds_ssp245.rlds) - rlut_ssp245.rlut

        if 'height' in list(nlw_hist.coords):
            nlw_total = xr.concat([nlw_hist, nlw_ssp245], dim='time').drop('height').rename('nlw')
        else:             
            nlw_total = xr.concat([nlw_hist, nlw_ssp245], dim='time').rename('nlw')
        nlw_globland_mean = nlw_total.where(landmask==landmask).weighted(weights).mean(['lon','lat']).rename('lw_globland')
        nlw_na = nlw_total.sel(lat=slice(0,65), lon=slice(-180,-80))
        nlw_wus = nlw_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
        nlw_wus_mean = nlw_wus.weighted(weights).mean(['lon','lat']).rename('lw_wus')
        nlw_avgs = xr.merge([nlw_globland_mean, nlw_wus_mean])#.assign_coords({'membermod':f'{model}_{member_id}'})
        model_lw_avg_list.append(nlw_avgs)
        model_lw_na_list.append(nlw_na)

    models_lw_avg_list.append(sum(model_lw_avg_list)/len(model_lw_avg_list))
    models_lw_na_list.append(sum(model_lw_na_list)/len(model_lw_na_list))
    #models_lw_avg_list[i] = (sum(model_lw_avg_list)/len(model_lw_avg_list))
    #models_lw_na_list[i] = (sum(model_lw_na_list)/len(model_lw_na_list))

CNRM-CM6-1 6
UKESM1-0-LL 6


In [298]:
models_lwd_avg_list = []
models_lwd_na_list = []


for model in list(modmembers_geq3_lwd.keys()):
#for (i, model) in [(2, 'CNRM-CM6-1'), (7,'UKESM1-0-LL')]:

    print(model, len(modmembers_lwd[model]))
    model_lwd_avg_list = []
    model_lwd_na_list = []
    

    for member_id in modmembers_lwd[model][:]:
        disk_hist = 'd4'
        disk_ssp245 = 'd4'
        rlds_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/rlds_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        rlds_ssp245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/rlds_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')

        if 'height' in list(rlds_hist.coords):
            lwd_total = xr.concat([rlds_hist, rlds_ssp245], dim='time').drop('height').rename({'rlds':'lwd'})
        else:             
            lwd_total = xr.concat([rlds_hist, rlds_ssp245], dim='time').rename({'rlds':'lwd'})
        lwd_globland_mean = lwd_total.where(landmask==landmask).weighted(weights).mean(['lon','lat']).rename({'lwd':'lwd_globland'})
        lwd_na = lwd_total.sel(lat=slice(0,65), lon=slice(-180,-80))
        lwd_wus = lwd_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
        lwd_wus_mean = lwd_wus.weighted(weights).mean(['lon','lat']).rename({'lwd':'lwd_wus'})
        lwd_avgs = xr.merge([lwd_globland_mean, lwd_wus_mean])#.assign_coords({'membermod':f'{model}_{member_id}'})
        model_lwd_avg_list.append(lwd_avgs)
        model_lwd_na_list.append(lwd_na)

    models_lwd_avg_list.append(sum(model_lwd_avg_list)/len(model_lwd_avg_list))
    models_lwd_na_list.append(sum(model_lwd_na_list)/len(model_lwd_na_list))
    #models_lw_avg_list[i] = (sum(model_lw_avg_list)/len(model_lw_avg_list))
    #models_lw_na_list[i] = (sum(model_lw_na_list)/len(model_lw_na_list))

GFDL-ESM4 2
IPSL-CM6A-LR 11
CNRM-CM6-1 6
MRI-ESM2-0 5
CNRM-ESM2-1 9
CanESM5 50
CanESM5-CanOE 3
UKESM1-0-LL 6
MIROC6 46
MPI-ESM1-2-LR 10
CESM2-WACCM 3
FGOALS-g3 4
MIROC-ES2L 3
NorESM2-LM 3
ACCESS-CM2 3
KACE-1-0-G 3
FIO-ESM-2-0 3
GISS-E2-1-G 14
EC-Earth3-Veg 2
EC-Earth3 6
CESM2 3


In [46]:
models_huss_avg_list = []
models_huss_na_list = []


for model in list(modmembers_geq3_huss.keys()):
#for (i, model) in [(2, 'CNRM-CM6-1'), (7,'UKESM1-0-LL')]:

    print(model, len(modmembers_huss[model]))
    model_huss_avg_list = []
    model_huss_na_list = []
    

    for member_id in modmembers_huss[model][:]:
        disk_hist = 'd4'
        disk_ssp245 = 'd4'
        huss_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/huss_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        huss_ssp245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/huss_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')

        if 'height' in list(huss_hist.coords):
            huss_total = xr.concat([huss_hist, huss_ssp245], dim='time').drop('height').rename({'huss':'huss'})
        else:             
            huss_total = xr.concat([huss_hist, huss_ssp245], dim='time').rename({'huss':'huss'})
        huss_globland_mean = huss_total.where(landmask==landmask).weighted(weights).mean(['lon','lat']).rename({'huss':'huss_globland'})
        huss_na = huss_total.sel(lat=slice(0,65), lon=slice(-180,-80))s
        huss_wus = huss_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
        huss_wus_mean = huss_wus.weighted(weights).mean(['lon','lat']).rename({'huss':'huss_wus'})
        huss_avgs = xr.merge([huss_globland_mean, huss_wus_mean])#.assign_coords({'membermod':f'{model}_{member_id}'})
        model_huss_avg_list.append(huss_avgs)
        model_huss_na_list.append(huss_na)

    models_huss_avg_list.append(sum(model_huss_avg_list)/len(model_huss_avg_list))
    models_huss_na_list.append(sum(model_huss_na_list)/len(model_huss_na_list))
    #models_lw_avg_list[i] = (sum(model_lw_avg_list)/len(model_lw_avg_list))
    #models_lw_na_list[i] = (sum(model_lw_na_list)/len(model_lw_na_list))

GFDL-ESM4 3
IPSL-CM6A-LR 11
CNRM-CM6-1 6
MRI-ESM2-0 5
CNRM-ESM2-1 9
CanESM5 50
CanESM5-CanOE 3
UKESM1-0-LL 13
MIROC6 50
MPI-ESM1-2-LR 10
CESM2-WACCM 3
FGOALS-g3 4
MIROC-ES2L 30
NorESM2-LM 3
ACCESS-CM2 3
KACE-1-0-G 3
FIO-ESM-2-0 3
GISS-E2-1-G 14
EC-Earth3-Veg 8
EC-Earth3 21
CESM2 3
EC-Earth3-Veg-LR 3


In [275]:
models_hus300_avg_list = []
models_hus300_na_list = []


for model in list(modmembers_geq3_hus300.keys()):
#for (i, model) in [(2, 'CNRM-CM6-1'), (7,'UKESM1-0-LL')]:

    print(model, len(modmembers_hus300[model]))
    model_hus300_avg_list = []
    model_hus300_na_list = []
    

    for member_id in modmembers_hus300[model][:]:
        disk_hist = 'd5'
        disk_ssp245 = 'd5'
        hus300_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/hus300_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        hus300_ssp245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/hus300_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')

        if ('height' in list(hus300_ssp245.coords)) & ('height' in list(hus300_ssp245.coords)):
            hus300_total = xr.concat([hus300_hist, hus300_ssp245], dim='time').drop('height').rename({'hus300':'hus300'})
        else:             
            hus300_total = xr.concat([hus300_hist, hus300_ssp245], dim='time').rename({'hus300':'hus300'})
        hus300_globland_mean = hus300_total.where(landmask==landmask).weighted(weights).mean(['lon','lat']).rename({'hus300':'hus300_globland'})
        hus300_na = hus300_total.sel(lat=slice(0,65), lon=slice(-180,-80))
        hus300_wus = hus300_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
        hus300_wus_mean = hus300_wus.weighted(weights).mean(['lon','lat']).rename({'hus300':'hus300_wus'})
        hus300_avgs = xr.merge([hus300_globland_mean, hus300_wus_mean])#.assign_coords({'membermod':f'{model}_{member_id}'})
        model_hus300_avg_list.append(hus300_avgs)
        model_hus300_na_list.append(hus300_na)

    models_hus300_avg_list.append(sum(model_hus300_avg_list)/len(model_hus300_avg_list))
    models_hus300_na_list.append(sum(model_hus300_na_list)/len(model_hus300_na_list))
    #models_lw_avg_list[i] = (sum(model_lw_avg_list)/len(model_lw_avg_list))
    #models_lw_na_list[i] = (sum(model_lw_na_list)/len(model_lw_na_list))

GFDL-ESM4 3
IPSL-CM6A-LR 11
CNRM-CM6-1 6
MRI-ESM2-0 5
CNRM-ESM2-1 9
CanESM5 50
CanESM5-CanOE 3
UKESM1-0-LL 6
MIROC6 50
MPI-ESM1-2-LR 10
CESM2-WACCM 3
FGOALS-g3 4
MIROC-ES2L 27
NorESM2-LM 3
ACCESS-CM2 3
KACE-1-0-G 3
FIO-ESM-2-0 3
GISS-E2-1-G 14
EC-Earth3-Veg 8
EC-Earth3 20
CESM2 3
EC-Earth3-Veg-LR 3


In [56]:
models_hus700_avg_list = []
models_hus700_na_list = []


for model in list(modmembers_geq3_hus700.keys()):
#for (i, model) in [(2, 'CNRM-CM6-1'), (7,'UKESM1-0-LL')]:

    print(model, len(modmembers_hus700[model]))
    model_hus700_avg_list = []
    model_hus700_na_list = []
    

    for member_id in modmembers_hus700[model][:]:
        disk_hist = 'd5'
        disk_ssp245 = 'd5'
        hus700_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/hus700_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        hus700_ssp245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/hus700_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')

        if ('height' in list(hus700_ssp245.coords)) & ('height' in list(hus700_ssp245.coords)):
            hus700_total = xr.concat([hus700_hist, hus700_ssp245], dim='time').drop('height').rename({'hus700':'hus700'})
        else:             
            hus700_total = xr.concat([hus700_hist, hus700_ssp245], dim='time').rename({'hus700':'hus700'})
        hus700_globland_mean = hus700_total.where(landmask==landmask).weighted(weights).mean(['lon','lat']).rename({'hus700':'hus700_globland'})
        hus700_na = hus700_total.sel(lat=slice(0,65), lon=slice(-180,-80))
        hus700_wus = hus700_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
        hus700_wus_mean = hus700_wus.weighted(weights).mean(['lon','lat']).rename({'hus700':'hus700_wus'})
        hus700_avgs = xr.merge([hus700_globland_mean, hus700_wus_mean])#.assign_coords({'membermod':f'{model}_{member_id}'})
        model_hus700_avg_list.append(hus700_avgs)
        model_hus700_na_list.append(hus700_na)

    models_hus700_avg_list.append(sum(model_hus700_avg_list)/len(model_hus700_avg_list))
    models_hus700_na_list.append(sum(model_hus700_na_list)/len(model_hus700_na_list))
    #models_lw_avg_list[i] = (sum(model_lw_avg_list)/len(model_lw_avg_list))
    #models_lw_na_list[i] = (sum(model_lw_na_list)/len(model_lw_na_list))

GFDL-ESM4 3
IPSL-CM6A-LR 11
CNRM-CM6-1 6
MRI-ESM2-0 5
CNRM-ESM2-1 9
CanESM5 50
CanESM5-CanOE 3
UKESM1-0-LL 6
MIROC6 50
MPI-ESM1-2-LR 10
CESM2-WACCM 3
FGOALS-g3 4
MIROC-ES2L 27
NorESM2-LM 3
ACCESS-CM2 3
KACE-1-0-G 3
FIO-ESM-2-0 3
GISS-E2-1-G 14
EC-Earth3-Veg 8
EC-Earth3 20
CESM2 3
EC-Earth3-Veg-LR 3


In [57]:
models_hus500_avg_list = []
models_hus500_na_list = []


for model in list(modmembers_geq3_hus500.keys()):
#for (i, model) in [(2, 'CNRM-CM6-1'), (7,'UKESM1-0-LL')]:

    print(model, len(modmembers_hus500[model]))
    model_hus500_avg_list = []
    model_hus500_na_list = []
    

    for member_id in modmembers_hus500[model][:]:
        disk_hist = 'd5'
        disk_ssp245 = 'd5'
        hus500_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/hus500_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        hus500_ssp245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/hus500_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')

        if ('height' in list(hus500_ssp245.coords)) & ('height' in list(hus500_ssp245.coords)):
            hus500_total = xr.concat([hus500_hist, hus500_ssp245], dim='time').drop('height').rename({'hus500':'hus500'})
        else:             
            hus500_total = xr.concat([hus500_hist, hus500_ssp245], dim='time').rename({'hus500':'hus500'})
        hus500_globland_mean = hus500_total.where(landmask==landmask).weighted(weights).mean(['lon','lat']).rename({'hus500':'hus500_globland'})
        hus500_na = hus500_total.sel(lat=slice(0,65), lon=slice(-180,-80))
        hus500_wus = hus500_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
        hus500_wus_mean = hus500_wus.weighted(weights).mean(['lon','lat']).rename({'hus500':'hus500_wus'})
        hus500_avgs = xr.merge([hus500_globland_mean, hus500_wus_mean])#.assign_coords({'membermod':f'{model}_{member_id}'})
        model_hus500_avg_list.append(hus500_avgs)
        model_hus500_na_list.append(hus500_na)

    models_hus500_avg_list.append(sum(model_hus500_avg_list)/len(model_hus500_avg_list))
    models_hus500_na_list.append(sum(model_hus500_na_list)/len(model_hus500_na_list))
    #models_lw_avg_list[i] = (sum(model_lw_avg_list)/len(model_lw_avg_list))
    #models_lw_na_list[i] = (sum(model_lw_na_list)/len(model_lw_na_list))

GFDL-ESM4 3
IPSL-CM6A-LR 11
CNRM-CM6-1 6
MRI-ESM2-0 5
CNRM-ESM2-1 9
CanESM5 50
CanESM5-CanOE 3
UKESM1-0-LL 6
MIROC6 50
MPI-ESM1-2-LR 10
CESM2-WACCM 3
FGOALS-g3 4
MIROC-ES2L 27
NorESM2-LM 3
ACCESS-CM2 3
KACE-1-0-G 3
FIO-ESM-2-0 3
GISS-E2-1-G 14
EC-Earth3-Veg 8
EC-Earth3 20
CESM2 3
EC-Earth3-Veg-LR 3


In [71]:
models_prw_avg_list = []
models_prw_na_list = []


for model in list(modmembers_geq3_prw.keys()):
#for (i, model) in [(2, 'CNRM-CM6-1'), (7,'UKESM1-0-LL')]:

    print(model, len(modmembers_prw[model]))
    model_prw_avg_list = []
    model_prw_na_list = []
    

    for member_id in modmembers_prw[model][:]:
        disk_hist = 'd5'
        disk_ssp245 = 'd5'
        prw_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/prw_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        prw_ssp245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/prw_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')

        if 'height' in list(prw_hist.coords):
            prw_total = xr.concat([prw_hist, prw_ssp245], dim='time').drop('height').rename({'prw':'prw'})
        else:             
            prw_total = xr.concat([prw_hist, prw_ssp245], dim='time').rename({'prw':'prw'})
        prw_globland_mean = prw_total.where(landmask==landmask).weighted(weights).mean(['lon','lat']).rename({'prw':'prw_globland'})
        prw_na = prw_total.sel(lat=slice(0,65), lon=slice(-180,-80))
        prw_wus = prw_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
        prw_wus_mean = prw_wus.weighted(weights).mean(['lon','lat']).rename({'prw':'prw_wus'})
        prw_avgs = xr.merge([prw_globland_mean, prw_wus_mean])#.assign_coords({'membermod':f'{model}_{member_id}'})
        model_prw_avg_list.append(prw_avgs)
        model_prw_na_list.append(prw_na)

    models_prw_avg_list.append(sum(model_prw_avg_list)/len(model_prw_avg_list))
    models_prw_na_list.append(sum(model_prw_na_list)/len(model_prw_na_list))
    #models_lw_avg_list[i] = (sum(model_lw_avg_list)/len(model_lw_avg_list))
    #models_lw_na_list[i] = (sum(model_lw_na_list)/len(model_lw_na_list))

EC-Earth3 13
CESM2 3
EC-Earth3-Veg-LR 3


In [207]:
models_hfls_avg_list = []
models_hfls_na_list = []


for model in list(modmembers_geq3_hfls.keys()):
#for (i, model) in [(2, 'CNRM-CM6-1'), (7,'UKESM1-0-LL')]:

    print(model, len(modmembers_hfls[model]))
    model_hfls_avg_list = []
    model_hfls_na_list = []
    

    for member_id in modmembers_hfls[model][:]:
        disk_hist = 'd4'
        disk_ssp245 = 'd4'
        hfls_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/hfls_Amon_{model}_historical_{member_id}_19400101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        hfls_ssp245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/hfls_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')

        if 'height' in list(hfls_hist.coords):
            hfls_total = xr.concat([hfls_hist, hfls_ssp245], dim='time').drop('height').rename({'hfls':'hfls'})
        else:             
            hfls_total = xr.concat([hfls_hist, hfls_ssp245], dim='time').rename({'hfls':'hfls'})
        hfls_globland_mean = hfls_total.where(landmask==landmask).weighted(weights).mean(['lon','lat']).rename({'hfls':'hfls_globland'})
        hfls_na = hfls_total.sel(lat=slice(0,65), lon=slice(-180,-80))
        hfls_wus = hfls_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
        hfls_wus_mean = hfls_wus.weighted(weights).mean(['lon','lat']).rename({'hfls':'hfls_wus'})
        hfls_avgs = xr.merge([hfls_globland_mean, hfls_wus_mean])#.assign_coords({'membermod':f'{model}_{member_id}'})
        model_hfls_avg_list.append(hfls_avgs)
        model_hfls_na_list.append(hfls_na)

    models_hfls_avg_list.append(sum(model_hfls_avg_list)/len(model_hfls_avg_list))
    models_hfls_na_list.append(sum(model_hfls_na_list)/len(model_hfls_na_list))
    #models_lw_avg_list[i] = (sum(model_lw_avg_list)/len(model_lw_avg_list))
    #models_lw_na_list[i] = (sum(model_lw_na_list)/len(model_lw_na_list))

GFDL-ESM4 1
IPSL-CM6A-LR 11
CNRM-CM6-1 6
MRI-ESM2-0 5
CNRM-ESM2-1 9
CanESM5 50
CanESM5-CanOE 3
UKESM1-0-LL 6
MIROC6 50
MPI-ESM1-2-LR 10
CESM2-WACCM 3
FGOALS-g3 4
MIROC-ES2L 4
NorESM2-LM 3
ACCESS-CM2 3
KACE-1-0-G 3
FIO-ESM-2-0 3
GISS-E2-1-G 14
EC-Earth3-Veg 8
EC-Earth3 22
CESM2 3
EC-Earth3-Veg-LR 3


In [257]:
models_e_avg_list = []
models_e_na_list = []


for model in list(modmembers_geq3_hfls.keys()):
#for (i, model) in [(2, 'CNRM-CM6-1'), (7,'UKESM1-0-LL')]:

    print(model, len(modmembers_hfls[model]))
    model_e_avg_list = []
    model_e_na_list = []
    
    for member_id in modmembers_hfls[model][:]:

        hfls_hist = xr.open_dataset(f'/d4/tessj/data/CMIP6/for_tess/{model}/hfls_Amon_{model}_historical_{member_id}_19400101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        hfls_ssp245 = xr.open_dataset(f'/d4/tessj/data/CMIP6/for_tess/{model}/hfls_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')
        disk_hist = modmembers_tas_hist_disk[model][member_id]
        disk_ssp245 = modmembers_tas_ssp245_disk[model][member_id]
        tas_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/tas_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        tas_ssp245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/tas_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')
        
        if 'height' in list(hfls_hist.coords):
            hfls_total = xr.concat([hfls_hist, hfls_ssp245], dim='time').drop('height').rename({'hfls':'hfls'})
        else:             
            hfls_total = xr.concat([hfls_hist, hfls_ssp245], dim='time').rename({'hfls':'hfls'})
        if 'height' in list(tas_hist.coords):            
            tas_total = xr.concat([tas_hist, tas_ssp245], dim='time').drop('height').rename({'tas':'tas'})
        else:
            tas_total = xr.concat([tas_hist, tas_ssp245], dim='time').rename({'tas':'tas'})
        
        e_total = (86400*1e-6)*(hfls_total.hfls)/(2.501-((2.361e-3)*(tas_total.tas - 273.15)))
        e_total = e_total.rename('e')

        e_globland_mean = e_total.where(landmask==landmask).weighted(weights).mean(['lon','lat']).rename('e_globland')
        e_na = e_total.sel(lat=slice(0,65), lon=slice(-180,-80))
        e_wus = e_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
        e_wus_mean = e_wus.weighted(weights).mean(['lon','lat']).rename('e_wus')
        e_avgs = xr.merge([e_globland_mean, e_wus_mean])#.assign_coords({'membermod':f'{model}_{member_id}'})
        model_e_avg_list.append(e_avgs)
        model_e_na_list.append(e_na)

    models_e_avg_list.append(sum(model_e_avg_list)/len(model_e_avg_list))
    models_e_na_list.append(sum(model_e_na_list)/len(model_e_na_list))
    #models_lw_avg_list[i] = (sum(model_lw_avg_list)/len(model_lw_avg_list))
    #models_lw_na_list[i] = (sum(model_lw_na_list)/len(model_lw_na_list))

GFDL-ESM4 1
IPSL-CM6A-LR 11
CNRM-CM6-1 6
MRI-ESM2-0 5
CNRM-ESM2-1 9
CanESM5 50
CanESM5-CanOE 3
UKESM1-0-LL 6
MIROC6 50
MPI-ESM1-2-LR 10
CESM2-WACCM 3
FGOALS-g3 4
MIROC-ES2L 4
NorESM2-LM 3
ACCESS-CM2 3
KACE-1-0-G 3
FIO-ESM-2-0 3
GISS-E2-1-G 14
EC-Earth3-Veg 8
EC-Earth3 22
CESM2 3
EC-Earth3-Veg-LR 3


In [13]:
sample_grid = xr.DataArray(coords = {'lat':(('lat'),np.arange(-89.5,89.6)),
                             'lon':(('lon'),np.arange(-179.5,179.6))},
                   dims = ['lat','lon'])

models_rldscs_avg_list = []
models_rldscs_na_list = []

for model in list(modmembers_rldscs.keys()):
    print(model, len(modmembers_rldscs[model]))
    model_rldscs_avg_list = []
    #model_tas_wus_list = []
    model_rldscs_na_list = []
    
    for member_id in modmembers_rldscs[model][:]:
        rldscs_hist = xr.open_dataset(f'/d5/tessj/data/CMIP6/cmip6-esgf-globus/historical-processed/{model}/rldscs_Amon_{model}_historical_{member_id}_19500101-20141231.nc').sel(time=slice('1950-01-01', '2014-12-30'))
        rldscs_ss245 = xr.open_dataset(f'/d5/tessj/data/CMIP6/cmip6-esgf-globus/ssp245-processed/{model}/rldscs_Amon_{model}_ssp245_{member_id}_20150101-20241231.nc')
        rldscs_total = xr.concat([rldscs_hist, rldscs_ss245], dim='time')
        rgrd = xe.Regridder(rldscs_total,sample_grid,'bilinear',ignore_degenerate=True, periodic=True)
        rldscs_total = rgrd(rldscs_total)
        #rldscs_total = fix_coords_lon_lat(rldscs_total)
        
        #landmask_model = landmask_era5.interp_like(rldscs_total)
        #landmask_model = landmask_model.where(landmask_model>0.4, np.nan)
        #landmask_model = landmask_model/landmask_model
        
        rldscs_globland_mean = rldscs_total.where(landmask==landmask).weighted(weights).mean(['lon','lat']).rename({'rldscs':'rldscs_globland'})
        rldscs_na = rldscs_total.sel(lat=slice(0,65), lon=slice(-180,-80))
        rldscs_wus = rldscs_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
        rldscs_wus_mean = rldscs_wus.weighted(weights).mean(['lon','lat']).rename({'rldscs':'rldscs_wus'})
        rldscs_avgs = xr.merge([rldscs_globland_mean, rldscs_wus_mean])#.assign_coords({'membermod':f'{model}_{member_id}'})
        model_rldscs_avg_list.append(rldscs_avgs)
        #model_rldscs_wus_list.append(rldscs_wus)
        model_rldscs_na_list.append(rldscs_na)

    models_rldscs_avg_list.append(sum(model_rldscs_avg_list)/len(model_rldscs_avg_list))
    #models_rldscs_wus_list.append(sum(model_rldscs_wus_list)/len(model_rldscs_wus_list))
    models_rldscs_na_list.append(sum(model_rldscs_na_list)/len(model_rldscs_na_list))


CNRM-CM6-1 6


/home/tessj/miniconda3/envs/ausfire/lib/python3.7/site-packages/xarray/core/dataarray.py:784: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  return key in self.data
/home/tessj/miniconda3/envs/ausfire/lib/python3.7/site-packages/xarray/core/dataarray.py:784: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  return key in self.data
/home/tessj/miniconda3/envs/ausfire/lib/python3.7/site-packages/xarray/core/dataarray.py:784: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  return key in self.data
/home/tessj/miniconda3/envs/ausfire/lib/python3.7/site-packages/xarray/core/dataarray.py:784: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  return key in self.data
/home/tessj/mini

IPSL-CM6A-LR 11


/home/tessj/miniconda3/envs/ausfire/lib/python3.7/site-packages/xarray/core/dataarray.py:784: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  return key in self.data
/home/tessj/miniconda3/envs/ausfire/lib/python3.7/site-packages/xarray/core/dataarray.py:784: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  return key in self.data
/home/tessj/miniconda3/envs/ausfire/lib/python3.7/site-packages/xarray/core/dataarray.py:784: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  return key in self.data
/home/tessj/miniconda3/envs/ausfire/lib/python3.7/site-packages/xarray/core/dataarray.py:784: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  return key in self.data
/home/tessj/mini

EC-Earth3-Veg 7


...

/home/tessj/miniconda3/envs/ausfire/lib/python3.7/site-packages/xarray/core/dataarray.py:784: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  return key in self.data
/home/tessj/miniconda3/envs/ausfire/lib/python3.7/site-packages/xarray/core/dataarray.py:784: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  return key in self.data
/home/tessj/miniconda3/envs/ausfire/lib/python3.7/site-packages/xarray/core/dataarray.py:784: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  return key in self.data
/home/tessj/miniconda3/envs/ausfire/lib/python3.7/site-packages/xarray/core/dataarray.py:784: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  return key in self.data
/home/tessj/mini

EC-Earth3-Veg-LR 3


/home/tessj/miniconda3/envs/ausfire/lib/python3.7/site-packages/xarray/core/dataarray.py:784: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  return key in self.data
/home/tessj/miniconda3/envs/ausfire/lib/python3.7/site-packages/xarray/core/dataarray.py:784: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  return key in self.data
/home/tessj/miniconda3/envs/ausfire/lib/python3.7/site-packages/xarray/core/dataarray.py:784: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  return key in self.data


GISS-E2-1-G 16


/home/tessj/miniconda3/envs/ausfire/lib/python3.7/site-packages/xarray/core/dataarray.py:784: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  return key in self.data
/home/tessj/miniconda3/envs/ausfire/lib/python3.7/site-packages/xarray/core/dataarray.py:784: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  return key in self.data
/home/tessj/miniconda3/envs/ausfire/lib/python3.7/site-packages/xarray/core/dataarray.py:784: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  return key in self.data
/home/tessj/miniconda3/envs/ausfire/lib/python3.7/site-packages/xarray/core/dataarray.py:784: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  return key in self.data
/home/tessj/mini

In [119]:
models_hfss_avg_list = []
models_hfss_na_list = []


for model in list(modmembers_geq3_hfss.keys()):
#for (i, model) in [(2, 'CNRM-CM6-1'), (7,'UKESM1-0-LL')]:

    print(model, len(modmembers_hfss[model]))
    model_hfss_avg_list = []
    model_hfss_na_list = []
    

    for member_id in modmembers_hfss[model][:]:
        disk_hist = 'd4'
        disk_ssp245 = 'd4'
        hfss_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/hfss_Amon_{model}_historical_{member_id}_19400101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        hfss_ssp245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/hfss_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')

        if 'height' in list(hfss_hist.coords):
            hfss_total = xr.concat([hfss_hist, hfss_ssp245], dim='time').drop('height').rename({'hfss':'hfss'})
        else:             
            hfss_total = xr.concat([hfss_hist, hfss_ssp245], dim='time').rename({'hfss':'hfss'})
        hfss_globland_mean = hfss_total.where(landmask==landmask).weighted(weights).mean(['lon','lat']).rename({'hfss':'hfss_globland'})
        hfss_na = hfss_total.sel(lat=slice(0,65), lon=slice(-180,-80))
        hfss_wus = hfss_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
        hfss_wus_mean = hfss_wus.weighted(weights).mean(['lon','lat']).rename({'hfss':'hfss_wus'})
        hfss_avgs = xr.merge([hfss_globland_mean, hfss_wus_mean])#.assign_coords({'membermod':f'{model}_{member_id}'})
        model_hfss_avg_list.append(hfss_avgs)
        model_hfss_na_list.append(hfss_na)

    models_hfss_avg_list.append(sum(model_hfss_avg_list)/len(model_hfss_avg_list))
    models_hfss_na_list.append(sum(model_hfss_na_list)/len(model_hfss_na_list))
    #models_lw_avg_list[i] = (sum(model_lw_avg_list)/len(model_lw_avg_list))
    #models_lw_na_list[i] = (sum(model_lw_na_list)/len(model_lw_na_list))

GFDL-ESM4 1
IPSL-CM6A-LR 11
CNRM-CM6-1 6
MRI-ESM2-0 5
CNRM-ESM2-1 9
CanESM5 50
CanESM5-CanOE 3
UKESM1-0-LL 6
MIROC6 47
MPI-ESM1-2-LR 10
CESM2-WACCM 3
FGOALS-g3 4
MIROC-ES2L 2
NorESM2-LM 3
ACCESS-CM2 3
KACE-1-0-G 3
FIO-ESM-2-0 3
GISS-E2-1-G 14
EC-Earth3-Veg 7
EC-Earth3 21
CESM2 3
EC-Earth3-Veg-LR 3


In [120]:
models_rsds_avg_list = []
models_rsds_na_list = []


for model in list(modmembers_geq3_rsds.keys()):
#for (i, model) in [(2, 'CNRM-CM6-1'), (7,'UKESM1-0-LL')]:

    print(model, len(modmembers_rsds[model]))
    model_rsds_avg_list = []
    model_rsds_na_list = []
    

    for member_id in modmembers_rsds[model][:]:
        disk_hist = 'd4'
        disk_ssp245 = 'd4'
        rsds_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/rsds_Amon_{model}_historical_{member_id}_19400101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        rsds_ssp245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/rsds_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')

        if 'height' in list(rsds_hist.coords):
            rsds_total = xr.concat([rsds_hist, rsds_ssp245], dim='time').drop('height').rename({'rsds':'rsds'})
        else:             
            rsds_total = xr.concat([rsds_hist, rsds_ssp245], dim='time').rename({'rsds':'rsds'})
        rsds_globland_mean = rsds_total.where(landmask==landmask).weighted(weights).mean(['lon','lat']).rename({'rsds':'rsds_globland'})
        rsds_na = rsds_total.sel(lat=slice(0,65), lon=slice(-180,-80))
        rsds_wus = rsds_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
        rsds_wus_mean = rsds_wus.weighted(weights).mean(['lon','lat']).rename({'rsds':'rsds_wus'})
        rsds_avgs = xr.merge([rsds_globland_mean, rsds_wus_mean])#.assign_coords({'membermod':f'{model}_{member_id}'})
        model_rsds_avg_list.append(rsds_avgs)
        model_rsds_na_list.append(rsds_na)

    models_rsds_avg_list.append(sum(model_rsds_avg_list)/len(model_rsds_avg_list))
    models_rsds_na_list.append(sum(model_rsds_na_list)/len(model_rsds_na_list))
    #models_lw_avg_list[i] = (sum(model_lw_avg_list)/len(model_lw_avg_list))
    #models_lw_na_list[i] = (sum(model_lw_na_list)/len(model_lw_na_list))

GFDL-ESM4 1
IPSL-CM6A-LR 11
CNRM-CM6-1 6
MRI-ESM2-0 5
CNRM-ESM2-1 9
CanESM5 50
CanESM5-CanOE 3
UKESM1-0-LL 6
MIROC6 48
MPI-ESM1-2-LR 10
CESM2-WACCM 3
FGOALS-g3 4
MIROC-ES2L 1
NorESM2-LM 3
ACCESS-CM2 3
KACE-1-0-G 3
FIO-ESM-2-0 3
GISS-E2-1-G 14
EC-Earth3-Veg 8
EC-Earth3 21
CESM2 3
EC-Earth3-Veg-LR 3


In [121]:
models_rsus_avg_list = []
models_rsus_na_list = []


for model in list(modmembers_geq3_rsus.keys()):
#for (i, model) in [(2, 'CNRM-CM6-1'), (7,'UKESM1-0-LL')]:

    print(model, len(modmembers_rsus[model]))
    model_rsus_avg_list = []
    model_rsus_na_list = []
    

    for member_id in modmembers_rsus[model][:]:
        disk_hist = 'd4'
        disk_ssp245 = 'd4'
        rsus_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/rsus_Amon_{model}_historical_{member_id}_19400101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        rsus_ssp245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/rsus_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')

        if 'height' in list(rsus_hist.coords):
            rsus_total = xr.concat([rsus_hist, rsus_ssp245], dim='time').drop('height').rename({'rsus':'rsus'})
        else:             
            rsus_total = xr.concat([rsus_hist, rsus_ssp245], dim='time').rename({'rsus':'rsus'})
        rsus_globland_mean = rsus_total.where(landmask==landmask).weighted(weights).mean(['lon','lat']).rename({'rsus':'rsus_globland'})
        rsus_na = rsus_total.sel(lat=slice(0,65), lon=slice(-180,-80))
        rsus_wus = rsus_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
        rsus_wus_mean = rsus_wus.weighted(weights).mean(['lon','lat']).rename({'rsus':'rsus_wus'})
        rsus_avgs = xr.merge([rsus_globland_mean, rsus_wus_mean])#.assign_coords({'membermod':f'{model}_{member_id}'})
        model_rsus_avg_list.append(rsus_avgs)
        model_rsus_na_list.append(rsus_na)

    models_rsus_avg_list.append(sum(model_rsus_avg_list)/len(model_rsus_avg_list))
    models_rsus_na_list.append(sum(model_rsus_na_list)/len(model_rsus_na_list))
    #models_lw_avg_list[i] = (sum(model_lw_avg_list)/len(model_lw_avg_list))
    #models_lw_na_list[i] = (sum(model_lw_na_list)/len(model_lw_na_list))

GFDL-ESM4 1
IPSL-CM6A-LR 11
CNRM-CM6-1 6
MRI-ESM2-0 5
CNRM-ESM2-1 9
CanESM5 50
CanESM5-CanOE 3
UKESM1-0-LL 6
MIROC6 49
MPI-ESM1-2-LR 10
CESM2-WACCM 3
FGOALS-g3 4
MIROC-ES2L 2
NorESM2-LM 2
ACCESS-CM2 3
KACE-1-0-G 3
FIO-ESM-2-0 3
GISS-E2-1-G 14
EC-Earth3-Veg 1
EC-Earth3 6
CESM2 3


In [122]:
models_rsdt_avg_list = []
models_rsdt_na_list = []


for model in list(modmembers_geq3_rsdt.keys()):
#for (i, model) in [(2, 'CNRM-CM6-1'), (7,'UKESM1-0-LL')]:

    print(model, len(modmembers_rsdt[model]))
    model_rsdt_avg_list = []
    model_rsdt_na_list = []
    

    for member_id in modmembers_rsdt[model][:]:
        disk_hist = 'd4'
        disk_ssp245 = 'd4'
        rsdt_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/rsdt_Amon_{model}_historical_{member_id}_19400101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        rsdt_ssp245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/rsdt_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')

        if 'height' in list(rsdt_hist.coords):
            rsdt_total = xr.concat([rsdt_hist, rsdt_ssp245], dim='time').drop('height').rename({'rsdt':'rsdt'})
        else:             
            rsdt_total = xr.concat([rsdt_hist, rsdt_ssp245], dim='time').rename({'rsdt':'rsdt'})
        rsdt_globland_mean = rsdt_total.where(landmask==landmask).weighted(weights).mean(['lon','lat']).rename({'rsdt':'rsdt_globland'})
        rsdt_na = rsdt_total.sel(lat=slice(0,65), lon=slice(-180,-80))
        rsdt_wus = rsdt_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
        rsdt_wus_mean = rsdt_wus.weighted(weights).mean(['lon','lat']).rename({'rsdt':'rsdt_wus'})
        rsdt_avgs = xr.merge([rsdt_globland_mean, rsdt_wus_mean])#.assign_coords({'membermod':f'{model}_{member_id}'})
        model_rsdt_avg_list.append(rsdt_avgs)
        model_rsdt_na_list.append(rsdt_na)

    models_rsdt_avg_list.append(sum(model_rsdt_avg_list)/len(model_rsdt_avg_list))
    models_rsdt_na_list.append(sum(model_rsdt_na_list)/len(model_rsdt_na_list))
    #models_lw_avg_list[i] = (sum(model_lw_avg_list)/len(model_lw_avg_list))
    #models_lw_na_list[i] = (sum(model_lw_na_list)/len(model_lw_na_list))

GFDL-ESM4 1
IPSL-CM6A-LR 11
CNRM-CM6-1 6
MRI-ESM2-0 5
CNRM-ESM2-1 9
CanESM5 50
CanESM5-CanOE 3
UKESM1-0-LL 6
MIROC6 50
MPI-ESM1-2-LR 10
CESM2-WACCM 3
FGOALS-g3 4
MIROC-ES2L 1
NorESM2-LM 3
ACCESS-CM2 3
KACE-1-0-G 3
FIO-ESM-2-0 3
GISS-E2-1-G 14
GISS-E2-1-H 5
EC-Earth3-Veg 8
EC-Earth3 21
CESM2 3
EC-Earth3-Veg-LR 3


In [123]:
models_rsut_avg_list = []
models_rsut_na_list = []


for model in list(modmembers_geq3_rsut.keys()):
#for (i, model) in [(2, 'CNRM-CM6-1'), (7,'UKESM1-0-LL')]:

    print(model, len(modmembers_rsut[model]))
    model_rsut_avg_list = []
    model_rsut_na_list = []
    

    for member_id in modmembers_rsut[model][:]:
        disk_hist = 'd4'
        disk_ssp245 = 'd4'
        rsut_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/rsut_Amon_{model}_historical_{member_id}_19400101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        rsut_ssp245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/rsut_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')

        if 'height' in list(rsut_hist.coords):
            rsut_total = xr.concat([rsut_hist, rsut_ssp245], dim='time').drop('height').rename({'rsut':'rsut'})
        else:             
            rsut_total = xr.concat([rsut_hist, rsut_ssp245], dim='time').rename({'rsut':'rsut'})
        rsut_globland_mean = rsut_total.where(landmask==landmask).weighted(weights).mean(['lon','lat']).rename({'rsut':'rsut_globland'})
        rsut_na = rsut_total.sel(lat=slice(0,65), lon=slice(-180,-80))
        rsut_wus = rsut_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
        rsut_wus_mean = rsut_wus.weighted(weights).mean(['lon','lat']).rename({'rsut':'rsut_wus'})
        rsut_avgs = xr.merge([rsut_globland_mean, rsut_wus_mean])#.assign_coords({'membermod':f'{model}_{member_id}'})
        model_rsut_avg_list.append(rsut_avgs)
        model_rsut_na_list.append(rsut_na)

    models_rsut_avg_list.append(sum(model_rsut_avg_list)/len(model_rsut_avg_list))
    models_rsut_na_list.append(sum(model_rsut_na_list)/len(model_rsut_na_list))
    #models_lw_avg_list[i] = (sum(model_lw_avg_list)/len(model_lw_avg_list))
    #models_lw_na_list[i] = (sum(model_lw_na_list)/len(model_lw_na_list))

GFDL-ESM4 1
IPSL-CM6A-LR 11
CNRM-CM6-1 6
MRI-ESM2-0 5
CNRM-ESM2-1 9
CanESM5 50
CanESM5-CanOE 3
UKESM1-0-LL 6
MIROC6 50
MPI-ESM1-2-LR 10
CESM2-WACCM 3
FGOALS-g3 4
MIROC-ES2L 2
NorESM2-LM 3
ACCESS-CM2 3
KACE-1-0-G 3
FIO-ESM-2-0 3
GISS-E2-1-G 14
GISS-E2-1-H 5
EC-Earth3-Veg 8
EC-Earth3 20
CESM2 3
EC-Earth3-Veg-LR 3


In [124]:
models_clt_avg_list = []
models_clt_na_list = []


for model in list(modmembers_geq3_clt.keys()):
#for (i, model) in [(2, 'CNRM-CM6-1'), (7,'UKESM1-0-LL')]:

    print(model, len(modmembers_clt[model]))
    model_clt_avg_list = []
    model_clt_na_list = []
    

    for member_id in modmembers_clt[model][:]:
        disk_hist = 'd4'
        disk_ssp245 = 'd4'
        clt_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/clt_Amon_{model}_historical_{member_id}_19400101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        clt_ssp245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/clt_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')

        if 'height' in list(clt_hist.coords):
            clt_total = xr.concat([clt_hist, clt_ssp245], dim='time').drop('height').rename({'clt':'clt'})
        else:             
            clt_total = xr.concat([clt_hist, clt_ssp245], dim='time').rename({'clt':'clt'})
        clt_globland_mean = clt_total.where(landmask==landmask).weighted(weights).mean(['lon','lat']).rename({'clt':'clt_globland'})
        clt_na = clt_total.sel(lat=slice(0,65), lon=slice(-180,-80))
        clt_wus = clt_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
        clt_wus_mean = clt_wus.weighted(weights).mean(['lon','lat']).rename({'clt':'clt_wus'})
        clt_avgs = xr.merge([clt_globland_mean, clt_wus_mean])#.assign_coords({'membermod':f'{model}_{member_id}'})
        model_clt_avg_list.append(clt_avgs)
        model_clt_na_list.append(clt_na)

    models_clt_avg_list.append(sum(model_clt_avg_list)/len(model_clt_avg_list))
    models_clt_na_list.append(sum(model_clt_na_list)/len(model_clt_na_list))
    #models_lw_avg_list[i] = (sum(model_lw_avg_list)/len(model_lw_avg_list))
    #models_lw_na_list[i] = (sum(model_lw_na_list)/len(model_lw_na_list))

GFDL-ESM4 1
IPSL-CM6A-LR 11
CNRM-CM6-1 6
MRI-ESM2-0 5
CNRM-ESM2-1 9
CanESM5 50
CanESM5-CanOE 3
UKESM1-0-LL 12
MIROC6 50
MPI-ESM1-2-LR 10
CESM2-WACCM 3
FGOALS-g3 4
MIROC-ES2L 30
NorESM2-LM 3
ACCESS-CM2 3
KACE-1-0-G 3
FIO-ESM-2-0 3
GISS-E2-1-G 14
EC-Earth3-Veg 8
EC-Earth3 20
CESM2 3
EC-Earth3-Veg-LR 3


In [1390]:
models_zg700_avg_list = []
models_zg700_wus_list = []

for model in list(modmembers_geq3_zg700.keys()):
    print(model, len(modmembers_zg700[model]))
    model_zg700_avg_list = []
    model_zg700_wus_list = []

    for member_id in modmembers_zg700[model][:]:
        disk_hist = 'd4'
        disk_ssp245 = 'd4'
        zg700_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/zg700_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        zg700_ss245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/zg700_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')
        if 'height' in list(zg700_hist.coords):
            zg700_total = xr.concat([zg700_hist, zg700_ss245], dim='time').drop('height')
        else:             
            zg700_total = xr.concat([zg700_hist, zg700_ss245], dim='time')
        zg700_globland_mean = zg700_total.where(landmask==landmask).weighted(weights).mean(['lon','lat']).rename({'zg700':'zg700_globland'})
        zg700_wus = zg700_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
        zg700_wus_mean = zg700_wus.weighted(weights).mean(['lon','lat']).rename({'zg700':'zg700_wus'})
        zg700_avgs = xr.merge([zg700_globland_mean, zg700_wus_mean])#.assign_coords({'membermod':f'{model}_{member_id}'})
        model_zg700_avg_list.append(zg700_avgs)
        model_zg700_wus_list.append(zg700_wus)
        
    models_zg700_avg_list.append(sum(model_zg700_avg_list)/len(model_zg700_avg_list))
    models_zg700_wus_list.append(sum(model_zg700_wus_list)/len(model_zg700_wus_list))


GFDL-ESM4 3
IPSL-CM6A-LR 11
CNRM-CM6-1 6
MRI-ESM2-0 5
CNRM-ESM2-1 9
CanESM5 50
CanESM5-CanOE 3
UKESM1-0-LL 6
MIROC6 50
MPI-ESM1-2-LR 9
CESM2-WACCM 3
FGOALS-g3 4
MIROC-ES2L 18
NorESM2-LM 3
ACCESS-CM2 3
KACE-1-0-G 3
FIO-ESM-2-0 3
GISS-E2-1-G 14
EC-Earth3-Veg 7
EC-Earth3 21
CESM2 3
EC-Earth3-Veg-LR 3


In [ ]:
models_zg700_na_list = []

for model in list(modmembers_geq3_zg700.keys()):
    print(model, len(modmembers_zg700[model]))
    model_zg700_na_list = []

    for member_id in modmembers_zg700[model][:]:
        disk_hist = 'd4'
        disk_ssp245 = 'd4'
        zg700_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/zg700_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        zg700_ss245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/zg700_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')
        if 'height' in list(zg700_hist.coords):
            zg700_total = xr.concat([zg700_hist, zg700_ss245], dim='time').drop('height')
        else:             
            zg700_total = xr.concat([zg700_hist, zg700_ss245], dim='time')
        zg700_na = zg700_total.sel(lat=slice(0,65), lon=slice(-180,-80))
        model_zg700_na_list.append(zg700_na)
        
    models_zg700_na_list.append(sum(model_zg700_na_list)/len(model_zg700_na_list))


## Calculate model seasonal trends as lists

In [116]:
models_zg700_na_trend_list_szn = []

for model in list(modmembers_geq3_zg700.keys()):
    print(model, len(modmembers_zg700[model]))
    model_zg700_na_trend_list_szn = {'JFM':[], 'AMJ':[], 'JAS':[], 'OND':[]}

    for member_id in modmembers_zg700[model][:]:
        #print(member_id)
        disk_hist = 'd4'
        disk_ssp245 = 'd4'
        zg700_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/zg700_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        zg700_ss245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/zg700_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')
        if 'height' in list(zg700_hist.coords):
            zg700_total = xr.concat([zg700_hist, zg700_ss245], dim='time').drop('height')
        else:             
            zg700_total = xr.concat([zg700_hist, zg700_ss245], dim='time')
        zg700_na = zg700_total.sel(lat=slice(0,65), lon=slice(-180,-80))
        
        for i, szn in enumerate(['JFM', 'AMJ', 'JAS', 'OND']):
            zg700_na_szn = zg700_na.resample(time='QS-JAN').mean('time').isel(time=slice(i,None,4))
            model_zg700_na_trend_list_szn[szn].append(zg700_na_szn.groupby('time.year').mean('time').sel(year=slice(1980,2024)).polyfit(dim='year', deg=1).sel(degree=1).zg700_polyfit_coefficients)
  
            #model_zg700_na_trend_list_szn[szn].append(zg700_na_trend_szn[szn])
    models_zg700_na_trend_list_szn.append(model_zg700_na_trend_list_szn)


GFDL-ESM4 3
IPSL-CM6A-LR 11
CNRM-CM6-1 6
MRI-ESM2-0 5
CNRM-ESM2-1 9
CanESM5 50
CanESM5-CanOE 3
UKESM1-0-LL 6
MIROC6 50
MPI-ESM1-2-LR 9
CESM2-WACCM 3
FGOALS-g3 4
MIROC-ES2L 18
NorESM2-LM 3
ACCESS-CM2 3
KACE-1-0-G 3
FIO-ESM-2-0 3
GISS-E2-1-G 14
EC-Earth3-Veg 7
EC-Earth3 21
CESM2 3
EC-Earth3-Veg-LR 3


In [186]:
models_tas_na_trend_list_szn = []

for model in list(modmembers_geq3_zg700.keys()):
    print(model, len(modmembers_zg700[model]))
    model_tas_na_trend_list_szn = {'JFM':[], 'AMJ':[], 'JAS':[], 'OND':[]}

    for member_id in modmembers_zg700[model][:]:
        #print(member_id)
        disk_hist = modmembers_tas_hist_disk[model][member_id]
        disk_ssp245 = modmembers_tas_ssp245_disk[model][member_id]
        tas_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/tas_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        tas_ss245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/tas_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')
        if 'height' in list(tas_hist.coords):
            tas_total = xr.concat([tas_hist, tas_ss245], dim='time').drop('height')
        else:             
            tas_total = xr.concat([tas_hist, tas_ss245], dim='time')
        tas_na = tas_total.sel(lat=slice(0,65), lon=slice(-180,-80))
        
        for i, szn in enumerate(['JFM', 'AMJ', 'JAS', 'OND']):
            tas_na_szn = tas_na.resample(time='QS-JAN').mean('time').isel(time=slice(i,None,4))
            model_tas_na_trend_list_szn[szn].append(tas_na_szn.groupby('time.year').mean('time').sel(year=slice(1980,2024)).polyfit(dim='year', deg=1).sel(degree=1).tas_polyfit_coefficients)
  
            #model_zg700_na_trend_list_szn[szn].append(zg700_na_trend_szn[szn])
    models_tas_na_trend_list_szn.append(model_tas_na_trend_list_szn)


GFDL-ESM4 3
IPSL-CM6A-LR 11
CNRM-CM6-1 6
MRI-ESM2-0 5
CNRM-ESM2-1 9
CanESM5 50
CanESM5-CanOE 3
UKESM1-0-LL 6
MIROC6 50
MPI-ESM1-2-LR 9
CESM2-WACCM 3
FGOALS-g3 4
MIROC-ES2L 18
NorESM2-LM 3
ACCESS-CM2 3
KACE-1-0-G 3
FIO-ESM-2-0 3
GISS-E2-1-G 14
EC-Earth3-Veg 7
EC-Earth3 21
CESM2 3
EC-Earth3-Veg-LR 3


In [262]:
models_tas_wus_trend_list_szn = []

for model in list(modmembers_geq3_tas.keys()):
    print(model, len(modmembers_tas[model]))
    model_tas_wus_trend_list_szn = {'JFM':[], 'AMJ':[], 'JAS':[], 'OND':[]}

    for member_id in modmembers_tas[model][:]:
        #print(member_id)
        disk_hist = modmembers_tas_hist_disk[model][member_id]
        disk_ssp245 = modmembers_tas_ssp245_disk[model][member_id]
        tas_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/tas_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        tas_ss245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/tas_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')
        if 'height' in list(tas_hist.coords):
            tas_total = xr.concat([tas_hist, tas_ss245], dim='time').drop('height')
        else:             
            tas_total = xr.concat([tas_hist, tas_ss245], dim='time')
        tas_wus = tas_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
        
        for i, szn in enumerate(['JFM', 'AMJ', 'JAS', 'OND']):
            tas_wus_szn = tas_wus.resample(time='QS-JAN').mean('time').isel(time=slice(i,None,4))
            model_tas_wus_trend_list_szn[szn].append(tas_wus_szn.groupby('time.year').mean('time').sel(year=slice(1980,2024)).polyfit(dim='year', deg=1).sel(degree=1).tas_polyfit_coefficients)
  
            #model_zg700_na_trend_list_szn[szn].append(zg700_na_trend_szn[szn])
    models_tas_wus_trend_list_szn.append(model_tas_wus_trend_list_szn)


GFDL-ESM4 3
IPSL-CM6A-LR 11
CNRM-CM6-1 6
MRI-ESM2-0 5
CNRM-ESM2-1 9
CanESM5 50
CanESM5-CanOE 3
UKESM1-0-LL 14
MIROC6 50
MPI-ESM1-2-LR 10
CESM2-WACCM 3
FGOALS-g3 4
MIROC-ES2L 30
NorESM2-LM 3
ACCESS-CM2 5
KACE-1-0-G 3
FIO-ESM-2-0 3
GISS-E2-1-G 14
GISS-E2-1-H 10
EC-Earth3-Veg 8
EC-Earth3 22
CESM2 3
EC-Earth3-Veg-LR 3
ACCESS-ESM1-5 10


In [263]:
models_huss_wus_trend_list_szn = []

for model in list(modmembers_geq3_huss.keys()):
    print(model, len(modmembers_huss[model]))
    model_huss_wus_trend_list_szn = {'JFM':[], 'AMJ':[], 'JAS':[], 'OND':[]}

    for member_id in modmembers_huss[model][:]:
        #print(member_id)
        disk_hist = 'd4'
        disk_ssp245 = 'd4'
        huss_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/huss_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        huss_ss245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/huss_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')
        if 'height' in list(huss_hist.coords):
            huss_total = xr.concat([huss_hist, huss_ss245], dim='time').drop('height')
        else:             
            huss_total = xr.concat([huss_hist, huss_ss245], dim='time')
        huss_wus = huss_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
        
        for i, szn in enumerate(['JFM', 'AMJ', 'JAS', 'OND']):
            huss_wus_szn = huss_wus.resample(time='QS-JAN').mean('time').isel(time=slice(i,None,4))
            model_huss_wus_trend_list_szn[szn].append(huss_wus_szn.groupby('time.year').mean('time').sel(year=slice(1980,2024)).polyfit(dim='year', deg=1).sel(degree=1).huss_polyfit_coefficients)
  
            #model_zg700_na_trend_list_szn[szn].append(zg700_na_trend_szn[szn])
    models_huss_wus_trend_list_szn.append(model_huss_wus_trend_list_szn)


GFDL-ESM4 3
IPSL-CM6A-LR 11
CNRM-CM6-1 6
MRI-ESM2-0 5
CNRM-ESM2-1 9
CanESM5 50
CanESM5-CanOE 3
UKESM1-0-LL 13
MIROC6 50
MPI-ESM1-2-LR 10
CESM2-WACCM 3
FGOALS-g3 4
MIROC-ES2L 30
NorESM2-LM 3
ACCESS-CM2 3
KACE-1-0-G 3
FIO-ESM-2-0 3
GISS-E2-1-G 14
EC-Earth3-Veg 8
EC-Earth3 21
CESM2 3
EC-Earth3-Veg-LR 3


In [ ]:
models_pr_wus_trend_list_szn = []

for model in list(modmembers_geq3_pr.keys()):
    print(model, len(modmembers_pr[model]))
    model_pr_wus_trend_list_szn = {'JFM':[], 'AMJ':[], 'JAS':[], 'OND':[]}

    for member_id in modmembers_pr[model][:]:
        #print(member_id)
        disk_hist = modmembers_pr_hist_disk[model][member_id]
        disk_ssp245 = modmembers_pr_ssp245_disk[model][member_id]
        pr_hist = xr.open_dapret(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/pr_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        pr_ss245 = xr.open_dapret(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/pr_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')
        if 'height' in list(pr_hist.coords):
            pr_total = xr.concat([pr_hist, pr_ss245], dim='time').drop('height')
        else:             
            pr_total = xr.concat([pr_hist, pr_ss245], dim='time')
        pr_wus = pr_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
        
        for i, szn in enumerate(['JFM', 'AMJ', 'JAS', 'OND']):
            pr_wus_szn = pr_wus.resample(time='QS-JAN').mean('time').isel(time=slice(i,None,4))
            model_pr_wus_trend_list_szn[szn].append(pr_wus_szn.groupby('time.year').mean('time').sel(year=slice(1980,2024)).polyfit(dim='year', deg=1).sel(degree=1).pr_polyfit_coefficients)
  
            #model_zg700_na_trend_list_szn[szn].append(zg700_na_trend_szn[szn])
    models_pr_wus_trend_list_szn.append(model_pr_wus_trend_list_szn)


In [24]:
models_zg700_nh_list = []

for model in list(modmembers_geq3_zg700.keys()):
    print(model, len(modmembers_zg700[model]))
    model_zg700_nh_list = []

    for member_id in modmembers_zg700[model][:]:
        disk_hist = 'd4'
        disk_ssp245 = 'd4'
        zg700_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/zg700_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        zg700_ss245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/zg700_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')
        if 'height' in list(zg700_hist.coords):
            zg700_total = xr.concat([zg700_hist, zg700_ss245], dim='time').drop('height')
        else:             
            zg700_total = xr.concat([zg700_hist, zg700_ss245], dim='time')
        zg700_nh = zg700_total.sel(lat=slice(0,90))
        model_zg700_nh_list.append(zg700_nh)
        
    models_zg700_nh_list.append(sum(model_zg700_nh_list)/len(model_zg700_nh_list))


GFDL-ESM4 3
IPSL-CM6A-LR 11
CNRM-CM6-1 6
MRI-ESM2-0 5
CNRM-ESM2-1 9
CanESM5 50
CanESM5-CanOE 3
UKESM1-0-LL 6
MIROC6 50
MPI-ESM1-2-LR 9
CESM2-WACCM 3
FGOALS-g3 4
MIROC-ES2L 18
NorESM2-LM 3
ACCESS-CM2 3
KACE-1-0-G 3
FIO-ESM-2-0 3
GISS-E2-1-G 14
EC-Earth3-Veg 7
EC-Earth3 21
CESM2 3
EC-Earth3-Veg-LR 3


In [ ]:
models_pr_avg_list = []#models_pr_avg_list[:-4]
#models_pr_wus_list = []#models_pr_wus_list[:-4]
models_pr_na_list = []

for model in list(modmembers_geq3_pr.keys()):
    print(model, len(modmembers_pr[model]))
    model_pr_avg_list = []
    #model_pr_wus_list = []
    model_pr_na_list = []
        
    for member_id in modmembers_pr[model][:]:
        disk_hist = modmembers_pr_hist_disk[model][member_id]
        disk_ssp245 = modmembers_pr_ssp245_disk[model][member_id]
        pr_hist = xr.open_dataset(f'/{disk_hist}/tessj/data/CMIP6/for_tess/{model}/pr_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc').sel(time=slice('1941-01-01', '2014-12-30'))
        pr_ss245 = xr.open_dataset(f'/{disk_ssp245}/tessj/data/CMIP6/for_tess/{model}/pr_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')
        if 'height' in list(pr_hist.coords):
            pr_total = xr.concat([pr_hist, pr_ss245], dim='time').drop('height')
        else:             
            pr_total = xr.concat([pr_hist, pr_ss245], dim='time')
        pr_globland_mean = pr_total.where(landmask==landmask).weighted(weights).mean(['lon','lat']).rename({'pr':'pr_globland'})
        pr_wus = pr_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
        pr_na = pr_total.sel(lat=slice(0,65), lon=slice(-180,-80))
        pr_wus_mean = pr_wus.weighted(weights).mean(['lon','lat']).rename({'pr':'pr_wus'})
        pr_avgs = xr.merge([pr_globland_mean, pr_wus_mean])#.assign_coords({'membermod':f'{model}_{member_id}'})
        model_pr_avg_list.append(pr_avgs)
        model_pr_na_list.append(pr_na)
        #model_pr_wus_list.append(pr_wus)
        
    models_pr_avg_list.append(sum(model_pr_avg_list)/len(model_pr_avg_list))
    #models_pr_wus_list.append(sum(model_pr_wus_list)/len(model_pr_wus_list))
    models_pr_na_list.append(sum(model_pr_na_list)/len(model_pr_na_list))


In [210]:
ds_ea_avg_list = []
for model in list(modmembers_VPD.keys())[:]:
    print(model, len(modmembers_VPD[model]))
    for member_id in modmembers_VPD[model]:
        ea_hist = xr.open_dataset(f'/d1/tessj/data/CMIP6/for_tess/{model}/ea_{model}_historical_{member_id}_18500101-20141231_regrid.nc').sel(time=slice('1970-01-01', '2014-12-30'))
        ea_ssp585 = xr.open_dataset(f'/d1/tessj/data/CMIP6/for_tess/{model}/ea_{model}_ssp585_{member_id}_20150101-20241231_regrid.nc')
        if 'height' in list(ea_hist.coords):
            ea_total = xr.concat([ea_hist, ea_ssp585], dim='time').drop('height')
        else:             
            ea_total = xr.concat([ea_hist, ea_ssp585], dim='time')
        ea_med = ea_total.where(landmask==landmask).sel(lon=slice(medbox[0],medbox[1]), lat=slice(medbox[2],medbox[3])).weighted(weights).mean(['lon','lat']).rename({'ea':'ea_med'})
        ea_e = ea_total.where(landmask==landmask).sel(lon=slice(ebox[0],ebox[1]), lat=slice(ebox[2],ebox[3])).weighted(weights).mean(['lon','lat']).rename({'ea':'ea_e'})
        ea_swc = ea_total.where(landmask==landmask).sel(lon=slice(swc_box[0],swc_box[1]), lat=slice(swc_box[2],swc_box[3])).weighted(weights).mean(['lon','lat']).rename({'ea':'ea_swc'})
        ea_swi = ea_total.where(landmask==landmask).sel(lon=slice(swi_box[0],swi_box[1]), lat=slice(swi_box[2],swi_box[3])).weighted(weights).mean(['lon','lat']).rename({'ea':'ea_swi'})
        ea_regions = xr.merge([ea_med,ea_e,ea_swc,ea_swi]).assign_coords({'membermod':f'{model}_{member_id}'})
        ds_ea_avg_list.append(ea_regions)
    

GFDL-CM4 1
GFDL-ESM4 1
CNRM-CM6-1 6
CNRM-ESM2-1 5
CanESM5 50
CanESM5-CanOE 3
UKESM1-0-LL 5
INM-CM4-8 1
MIROC6 50
MPI-ESM1-2-HR 2
INM-CM5-0 1
MPI-ESM1-2-LR 10
FGOALS-g3 4
MIROC-ES2L 10
IPSL-CM6A-LR 6
KACE-1-0-G 3
FGOALS-f3-L 1
MRI-ESM2-0 2
NorESM2-LM 1
NorESM2-MM 1
CNRM-CM6-1-HR 1
HadGEM3-GC31-LL 4
GISS-E2-1-G 6
EC-Earth3-Veg 7
EC-Earth3 58
CESM2-WACCM 3
ACCESS-CM2 3
HadGEM3-GC31-MM 4
CESM2 3
CMCC-CM2-SR5 1
FIO-ESM-2-0 3
IITM-ESM 1
EC-Earth3-Veg-LR 3
CAS-ESM2-0 2
EC-Earth3-CC 1
CMCC-ESM2 1
ACCESS-ESM1-5 10
KIOST-ESM 1


In [14]:
datetimes_np64 = models_rldscs_na_list[2].time.values

In [ ]:
models_tas_wus_list = models_tas_na_list
models_pr_wus_list = models_pr_na_list

## Fix datetime of avg timeseries and trends

In [ ]:
ds_tas_avg_list_fixdt = []

for i, ds in enumerate(models_tas_avg_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('tas incomplete' + list(modmembers_geq3_tas.keys())[i])
    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        ds_tas_avg_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_tas.keys())[i]}))
    else:
        ds_tas_avg_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_tas.keys())[i]}))

ds_tas_wus_list_fixdt = []

for i, ds in enumerate(models_tas_wus_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('tas incomplete' + list(modmembers_geq3_tas.keys())[i])
    
    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_tas_wus_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_tas.keys())[i]}))
    else:
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_tas_wus_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_tas.keys())[i]}))

In [ ]:
ds_pr_avg_list_fixdt = []

for i, ds in enumerate(models_pr_avg_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('pr incomplete' + list(modmembers_geq3_pr.keys())[i])
    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        ds_pr_avg_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_pr.keys())[i]}))
    else:
        ds_pr_avg_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_pr.keys())[i]}))

ds_pr_wus_list_fixdt = []

for i, ds in enumerate(models_pr_wus_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('pr incomplete' + list(modmembers_geq3_pr.keys())[i])

    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_pr_wus_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_pr.keys())[i]}))
    else:
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_pr_wus_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_pr.keys())[i]}))

In [15]:
ds_rldscs_avg_list_fixdt = []

for i, ds in enumerate(models_rldscs_avg_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('rldscs incomplete' + list(modmembers_rldscs.keys())[i])
    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        ds_rldscs_avg_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_rldscs.keys())[i]}))
    else:
        ds_rldscs_avg_list_fixdt.append(ds.assign_coords({'model':list(modmembers_rldscs.keys())[i]}))

ds_rldscs_na_list_fixdt = []

for i, ds in enumerate(models_rldscs_na_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('rldscs incomplete' + list(modmembers_geq3_rldscs.keys())[i])
    
    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        if 'nbnd' in list(ds.coords):
            ds = ds.drop('nbnd')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})

                
        ds_rldscs_wus_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_rldscs.keys())[i]}))
    else:
        if 'nbnd' in list(ds.coords):
            ds = ds.drop('nbnd')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        #rgrd = xe.Regridder(ds,sample_grid,'bilinear',ignore_degenerate=True, periodic=True)
        #ds = rgrd(ds)
        ds_rldscs_na_list_fixdt.append(ds.assign_coords({'model':list(modmembers_rldscs.keys())[i]}))

In [1500]:
ds_zg700_avg_list_fixdt = []

for i, ds in enumerate(models_zg700_avg_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('zg700 incomplete' + list(modmembers_geq3_zg700.keys())[i])
    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        ds_zg700_avg_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_zg700.keys())[i]}))
    else:
        ds_zg700_avg_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_zg700.keys())[i]}))

ds_zg700_wus_list_fixdt = []

for i, ds in enumerate(models_zg700_wus_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('zg700 incomplete' + list(modmembers_geq3_zg700.keys())[i])

    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_zg700_wus_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_zg700.keys())[i]}))
    else:
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_zg700_wus_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_zg700.keys())[i]}))

In [ ]:
        
ds_zg700_na_list_fixdt = []

for i, ds in enumerate(models_zg700_na_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('zg700 incomplete' + list(modmembers_geq3_zg700.keys())[i])

    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_zg700_na_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_zg700.keys())[i]}))
    else:
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_zg700_na_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_zg700.keys())[i]}))

In [30]:
        
ds_zg700_nh_list_fixdt = []

for i, ds in enumerate(models_zg700_nh_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('zg700 incomplete' + list(modmembers_geq3_zg700.keys())[i])

    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_zg700_nh_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_zg700.keys())[i]}))
    else:
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_zg700_nh_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_zg700.keys())[i]}))

In [91]:
ds_lw_avg_list_fixdt = []

for i, ds in enumerate(models_lw_avg_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('lw incomplete' + list(modmembers_geq3_lw.keys())[i])
    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        ds_lw_avg_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_lw.keys())[i]}))
    else:
        ds_lw_avg_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_lw.keys())[i]}))

        
ds_lw_na_list_fixdt = []

for i, ds in enumerate(models_lw_na_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('lw incomplete' + list(modmembers_geq3_lw.keys())[i])

    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_lw_na_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_lw.keys())[i]}))
    else:
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_lw_na_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_lw.keys())[i]}))

In [299]:
ds_lwd_avg_list_fixdt = []

for i, ds in enumerate(models_lwd_avg_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('lwd incomplete' + list(modmembers_geq3_lwd.keys())[i])
    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        ds_lwd_avg_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_lwd.keys())[i]}))
    else:
        ds_lwd_avg_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_lwd.keys())[i]}))

        
ds_lwd_na_list_fixdt = []

for i, ds in enumerate(models_lwd_na_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('lwd incomplete' + list(modmembers_geq3_lwd.keys())[i])

    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_lwd_na_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_lwd.keys())[i]}))
    else:
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_lwd_na_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_lwd.keys())[i]}))

In [76]:
ds_huss_avg_list_fixdt = []

for i, ds in enumerate(models_huss_avg_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('huss incomplete' + list(modmembers_geq3_huss.keys())[i])
    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        ds_huss_avg_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_huss.keys())[i]}))
    else:
        ds_huss_avg_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_huss.keys())[i]}))

        
ds_huss_na_list_fixdt = []

for i, ds in enumerate(models_huss_na_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('huss incomplete' + list(modmembers_geq3_huss.keys())[i])

    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_huss_na_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_huss.keys())[i]}))
    else:
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_huss_na_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_huss.keys())[i]}))

In [276]:
ds_hus300_avg_list_fixdt = []

for i, ds in enumerate(models_hus300_avg_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('hus300 incomplete' + list(modmembers_geq3_hus300.keys())[i])
    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        ds_hus300_avg_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_hus300.keys())[i]}))
    else:
        ds_hus300_avg_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_hus300.keys())[i]}))

        
ds_hus300_na_list_fixdt = []

for i, ds in enumerate(models_hus300_na_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('hus300 incomplete' + list(modmembers_geq3_hus300.keys())[i])

    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_hus300_na_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_hus300.keys())[i]}))
    else:
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_hus300_na_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_hus300.keys())[i]}))

In [61]:
ds_hus500_avg_list_fixdt = []

for i, ds in enumerate(models_hus500_avg_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('hus500 incomplete' + list(modmembers_geq3_hus500.keys())[i])
    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        ds_hus500_avg_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_hus500.keys())[i]}))
    else:
        ds_hus500_avg_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_hus500.keys())[i]}))

        
ds_hus500_na_list_fixdt = []

for i, ds in enumerate(models_hus500_na_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('hus500 incomplete' + list(modmembers_geq3_hus500.keys())[i])

    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_hus500_na_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_hus500.keys())[i]}))
    else:
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_hus500_na_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_hus500.keys())[i]}))

In [62]:
ds_hus700_avg_list_fixdt = []

for i, ds in enumerate(models_hus700_avg_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('hus700 incomplete' + list(modmembers_geq3_hus700.keys())[i])
    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        ds_hus700_avg_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_hus700.keys())[i]}))
    else:
        ds_hus700_avg_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_hus700.keys())[i]}))

        
ds_hus700_na_list_fixdt = []

for i, ds in enumerate(models_hus700_na_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('hus700 incomplete' + list(modmembers_geq3_hus700.keys())[i])

    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_hus700_na_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_hus700.keys())[i]}))
    else:
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_hus700_na_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_hus700.keys())[i]}))

In [79]:
ds_prw_avg_list_fixdt = []

for i, ds in enumerate(models_prw_avg_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('prw incomplete' + list(modmembers_geq3_prw.keys())[i])
    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        ds_prw_avg_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_prw.keys())[i]}))
    else:
        ds_prw_avg_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_prw.keys())[i]}))

        
ds_prw_na_list_fixdt = []

for i, ds in enumerate(models_prw_na_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('prw incomplete' + list(modmembers_geq3_prw.keys())[i])

    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_prw_na_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_prw.keys())[i]}))
    else:
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_prw_na_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_prw.keys())[i]}))

In [128]:
ds_hfls_avg_list_fixdt = []

for i, ds in enumerate(models_hfls_avg_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('hfls incomplete' + list(modmembers_geq3_hfls.keys())[i])
    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        ds_hfls_avg_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_hfls.keys())[i]}))
    else:
        ds_hfls_avg_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_hfls.keys())[i]}))

        
ds_hfls_na_list_fixdt = []

for i, ds in enumerate(models_hfls_na_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('hfls incomplete' + list(modmembers_geq3_hfls.keys())[i])

    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_hfls_na_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_hfls.keys())[i]}))
    else:
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_hfls_na_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_hfls.keys())[i]}))

In [336]:
ds_e_avg_list_fixdt = []

for i, ds in enumerate(models_e_avg_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('e incomplete' + list(modmembers_geq3_hfls.keys())[i])
    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        ds_e_avg_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_hfls.keys())[i]}))
    else:
        ds_e_avg_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_hfls.keys())[i]}))

        
ds_e_na_list_fixdt = []

for i, ds in enumerate(models_e_na_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('e incomplete' + list(modmembers_geq3_hfls.keys())[i])

    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_e_na_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_hfls.keys())[i]}))
    else:
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_e_na_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_hfls.keys())[i]}))

In [129]:
ds_hfss_avg_list_fixdt = []

for i, ds in enumerate(models_hfss_avg_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('hfss incomplete' + list(modmembers_geq3_hfss.keys())[i])
    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        ds_hfss_avg_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_hfss.keys())[i]}))
    else:
        ds_hfss_avg_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_hfss.keys())[i]}))

        
ds_hfss_na_list_fixdt = []

for i, ds in enumerate(models_hfss_na_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('hfss incomplete' + list(modmembers_geq3_hfss.keys())[i])

    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_hfss_na_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_hfss.keys())[i]}))
    else:
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_hfss_na_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_hfss.keys())[i]}))

In [130]:
ds_rsds_avg_list_fixdt = []

for i, ds in enumerate(models_rsds_avg_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('rsds incomplete' + list(modmembers_geq3_rsds.keys())[i])
    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        ds_rsds_avg_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_rsds.keys())[i]}))
    else:
        ds_rsds_avg_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_rsds.keys())[i]}))

        
ds_rsds_na_list_fixdt = []

for i, ds in enumerate(models_rsds_na_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('rsds incomplete' + list(modmembers_geq3_rsds.keys())[i])

    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_rsds_na_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_rsds.keys())[i]}))
    else:
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_rsds_na_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_rsds.keys())[i]}))

In [131]:
ds_rsus_avg_list_fixdt = []

for i, ds in enumerate(models_rsus_avg_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('rsus incomplete' + list(modmembers_geq3_rsus.keys())[i])
    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        ds_rsus_avg_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_rsus.keys())[i]}))
    else:
        ds_rsus_avg_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_rsus.keys())[i]}))

        
ds_rsus_na_list_fixdt = []

for i, ds in enumerate(models_rsus_na_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('rsus incomplete' + list(modmembers_geq3_rsus.keys())[i])

    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_rsus_na_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_rsus.keys())[i]}))
    else:
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_rsus_na_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_rsus.keys())[i]}))

In [132]:
ds_rsdt_avg_list_fixdt = []

for i, ds in enumerate(models_rsdt_avg_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('rsdt incomplete' + list(modmembers_geq3_rsdt.keys())[i])
    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        ds_rsdt_avg_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_rsdt.keys())[i]}))
    else:
        ds_rsdt_avg_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_rsdt.keys())[i]}))

        
ds_rsdt_na_list_fixdt = []

for i, ds in enumerate(models_rsdt_na_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('rsdt incomplete' + list(modmembers_geq3_rsdt.keys())[i])

    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_rsdt_na_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_rsdt.keys())[i]}))
    else:
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_rsdt_na_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_rsdt.keys())[i]}))

In [133]:
ds_rsut_avg_list_fixdt = []

for i, ds in enumerate(models_rsut_avg_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('rsut incomplete' + list(modmembers_geq3_rsut.keys())[i])
    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        ds_rsut_avg_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_rsut.keys())[i]}))
    else:
        ds_rsut_avg_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_rsut.keys())[i]}))

        
ds_rsut_na_list_fixdt = []

for i, ds in enumerate(models_rsut_na_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('rsut incomplete' + list(modmembers_geq3_rsut.keys())[i])

    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_rsut_na_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_rsut.keys())[i]}))
    else:
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_rsut_na_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_rsut.keys())[i]}))

In [134]:
ds_clt_avg_list_fixdt = []

for i, ds in enumerate(models_clt_avg_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('clt incomplete' + list(modmembers_geq3_clt.keys())[i])
    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        ds_clt_avg_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_clt.keys())[i]}))
    else:
        ds_clt_avg_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_clt.keys())[i]}))

        
ds_clt_na_list_fixdt = []

for i, ds in enumerate(models_clt_na_list):
    if len(ds.time.values) < len(datetimes_np64):
        print('clt incomplete' + list(modmembers_geq3_clt.keys())[i])

    elif str(type(ds.time.values[0])) != "<class 'numpy.datetime64'>":
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_clt_na_list_fixdt.append(ds.assign_coords({'time':datetimes_np64}).assign_coords({'model':list(modmembers_geq3_clt.keys())[i]}))
    else:
        if 'bnds' in list(ds.coords):
            ds = ds.drop('bnds')
        if 'time_bounds' in list(ds.coords):
            ds = ds.rename({'time_bounds':'time_bnds'})
        ds_clt_na_list_fixdt.append(ds.assign_coords({'model':list(modmembers_geq3_clt.keys())[i]}))

## Concatenate and save to file

In [85]:
ds_pr_avg = xr.concat(ds_pr_avg_list_fixdt, dim='model')
ds_pr_wus = xr.concat(ds_pr_wus_list_fixdt, dim='model')


In [92]:
ds_lw_avg = xr.concat(ds_lw_avg_list_fixdt, dim='model')
ds_lw_wus = xr.concat(ds_lw_na_list_fixdt, dim='model')


In [94]:
ds_lw_avg.to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_lw_avgs.nc')
ds_lw_wus.to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_lw_na.nc')


In [301]:
ds_lwd_avg = xr.concat(ds_lwd_avg_list_fixdt, dim='model')
ds_lwd_wus = xr.concat(ds_lwd_na_list_fixdt, dim='model')


In [308]:
ds_lwd_avg.to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_lwd_avgs.nc')
ds_lwd_wus.drop('time_bnds').to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_lwd_na.nc')


In [92]:
ds_lw_avg = xr.concat(ds_lw_avg_list_fixdt, dim='model')
ds_lw_wus = xr.concat(ds_lw_na_list_fixdt, dim='model')


In [16]:
ds_rldscs_avg = xr.concat(ds_rldscs_avg_list_fixdt, dim='model')
ds_rldscs_wus = xr.concat(ds_rldscs_na_list_fixdt, dim='model', combine_attrs='drop')


In [17]:
ds_rldscs_avg.to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_rldscs_avgs.nc')
ds_rldscs_wus.drop('time_bnds').to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_rldscs_na.nc')


In [277]:
ds_hus300_avg = xr.concat(ds_hus300_avg_list_fixdt, dim='model')
ds_hus300_wus = xr.concat(ds_hus300_na_list_fixdt, dim='model')


In [278]:
ds_hus300_avg.to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_hus300_avgs.nc')
ds_hus300_wus.drop('time_bnds').to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_hus300_na.nc')


In [63]:
ds_hus500_avg = xr.concat(ds_hus500_avg_list_fixdt, dim='model')
ds_hus500_wus = xr.concat(ds_hus500_na_list_fixdt, dim='model')


In [64]:
ds_hus500_avg.to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_hus500_avgs.nc')
ds_hus500_wus.drop('time_bnds').to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_hus500_na.nc')


In [65]:
ds_hus700_avg = xr.concat(ds_hus700_avg_list_fixdt, dim='model')
ds_hus700_wus = xr.concat(ds_hus700_na_list_fixdt, dim='model')


In [66]:
ds_hus700_avg.to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_hus700_avgs.nc')
ds_hus700_wus.drop('time_bnds').to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_hus700_na.nc')


In [80]:
ds_huss_avg = xr.concat(ds_huss_avg_list_fixdt, dim='model')
ds_huss_wus = xr.concat(ds_huss_na_list_fixdt, dim='model')


In [469]:
ds_huss_avg.to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_huss_avgs.nc')
ds_huss_wus.drop('time_bnds').to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_huss_na.nc')


In [81]:
ds_prw_avg = xr.concat(ds_prw_avg_list_fixdt, dim='model')
ds_prw_wus = xr.concat(ds_prw_na_list_fixdt, dim='model')


In [84]:
ds_prw_avg.to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_prw_avgs.nc')
ds_prw_wus.drop('time_bnds').to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_prw_na.nc')


In [136]:
ds_hfls_avg = xr.concat(ds_hfls_avg_list_fixdt, dim='model')
ds_hfls_wus = xr.concat(ds_hfls_na_list_fixdt, dim='model')


In [137]:
ds_hfls_avg.to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_hfls_avgs.nc')
ds_hfls_wus.drop('time_bnds').to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_hfls_na.nc')


In [338]:
ds_e_avg = xr.concat(ds_e_avg_list_fixdt, dim='model')
ds_e_wus = xr.concat(ds_e_na_list_fixdt, dim='model')


In [340]:
ds_e_avg.to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_e_avgs.nc')
ds_e_wus.to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_e_na.nc')


In [138]:
ds_hfss_avg = xr.concat(ds_hfss_avg_list_fixdt, dim='model')
ds_hfss_wus = xr.concat(ds_hfss_na_list_fixdt, dim='model')


In [139]:
ds_hfss_avg.to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_hfss_avgs.nc')
ds_hfss_wus.drop('time_bnds').to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_hfss_na.nc')


In [140]:
ds_rsds_avg = xr.concat(ds_rsds_avg_list_fixdt, dim='model')
ds_rsds_wus = xr.concat(ds_rsds_na_list_fixdt, dim='model')


In [141]:
ds_rsds_avg.to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_rsds_avgs.nc')
ds_rsds_wus.drop('time_bnds').to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_rsds_na.nc')


In [142]:
ds_rsus_avg = xr.concat(ds_rsus_avg_list_fixdt, dim='model')
ds_rsus_wus = xr.concat(ds_rsus_na_list_fixdt, dim='model')


In [143]:
ds_rsus_avg.to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_rsus_avgs.nc')
ds_rsus_wus.drop('time_bnds').to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_rsus_na.nc')


In [144]:
ds_rsdt_avg = xr.concat(ds_rsdt_avg_list_fixdt, dim='model')
ds_rsdt_wus = xr.concat(ds_rsdt_na_list_fixdt, dim='model')


In [145]:
ds_rsdt_avg.to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_rsdt_avgs.nc')
ds_rsdt_wus.drop('time_bnds').to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_rsdt_na.nc')


In [146]:
ds_rsut_avg = xr.concat(ds_rsut_avg_list_fixdt, dim='model')
ds_rsut_wus = xr.concat(ds_rsut_na_list_fixdt, dim='model')


In [147]:
ds_rsut_avg.to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_rsut_avgs.nc')
ds_rsut_wus.drop('time_bnds').to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_rsut_na.nc')


In [148]:
ds_clt_avg = xr.concat(ds_clt_avg_list_fixdt, dim='model')
ds_clt_wus = xr.concat(ds_clt_na_list_fixdt, dim='model')


In [149]:
ds_clt_avg.to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_clt_avgs.nc')
ds_clt_wus.drop('time_bnds').to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_clt_na.nc')


In [88]:
#ds_zg700_avg = xr.concat(ds_zg700_avg_list_fixdt, dim='model')
ds_zg700_wus = xr.concat(ds_zg700_na_list_fixdt, dim='model')


In [31]:
ds_zg700_nh = xr.concat(ds_zg700_nh_list_fixdt, dim='model')
ds_zg700_nh.drop('time_bnds').to_netcdf('/home/tessj/wus_temp_trends/cmip6_ensmean_zg700_nh.nc')
